# D4-ORQB — Model IV

This is the complete, readable notebook implementation of the `dev` branch
D4 Orbit-Reuploading Quantum Bottleneck (D4-ORQB). It embeds the configuration,
data pipeline, Model-IV audit, D4 lifting, morphology features, shared MBConv
encoder, TorchQuantum circuit, hybrid classifier, metrics, checkpointing, and
training engine. It does **not** call the package CLI or hide the implementation
behind `src/` imports.

Source snapshot: `8221df807cb9`. The embedded source cells record SHA-256 hashes in
notebook metadata. Package-relative imports are removed only because all
definitions share this notebook namespace. The package-only `__init__.py`
re-exports and `main.py` argparse shell are replaced by the direct, visible
orchestration cells below; every computational module is embedded in full.

## Architecture

```text
.npy image [B,1,H,W]
  -> explicit eight-view D4 lift [B,8,1,H,W]
  -> deterministic 8-channel morphology per view [B,8,8,H,W]
  -> one shared compact MBConv encoder [B,8,128]
  -> orbit projection and bounded angles [B,4,2,8]
  -> 8-qubit, 4-head, 2-reupload TorchQuantum D4 orbit circuit
  -> 48 invariant observables
  -> three-class head (axion / cdm / no_sub)
```

The classical context stage has **276,993** trainable parameters; the
quantum stage has **249,409**. The circuit itself always has exactly
88 trainable parameters and 48 invariant outputs. The classical checkpoint
initializes the source-defined backbone for the quantum stage; the quantum core
and main classifier `head` are fresh.

## Dataset contract

The supplied `val/` directory remains development validation. A fixed 15% class-stratified test holdout is carved from `train/`, leaving 85% for training. The Model-IV signal audit sees only that 85% training partition plus supplied validation; it never sees the carved test samples. This is a file-level split because the supplied layout exposes no source/pair grouping manifest.

| Field | Rule |
| --- | --- |
| `DEVELOPMENT_ROOT` | Required `Model_IV/train` class-folder root |
| `VALIDATION_ROOT` | Required supplied `Model_IV/val` class-folder root |
| `TEST_ROOT` | Keep empty; test is carved from development |
| `CACHE_ROOT` | Required writable cache root |
| `OUTPUT_DIR` | Required fresh directory for training |

Every split is persisted, class-stratified with seed 42, checked for full index
coverage, and checked for model-visible SHA-256 overlap. Validation alone selects
checkpoints and controls early stopping. The validation-selected checkpoint is
reloaded from `best.pt` for final testing; test metrics never feed back into training.


Model IV keeps the existing foreground-suppressed morphology bank, the
1,395-feature invariant physics summary, and the CPU-only integrity/signal gate.
Its source checkpoint prefix also transfers the auxiliary 1,395→3 summary
classifier; the TorchQuantum core and main `head` are still initialized fresh.
These are source behavior, not notebook inventions.


## How to run

1. Install `src/requirements.txt` (TorchQuantum is pinned to commit
   `8dc3255c51477dd4c28892049571df032c77e2ff`).
2. Fill only the blank runtime path fields in the next code cell.
3. Run through architecture verification, then the two training stages.
4. Review development-validation evidence.
5. Set `CONFIRM_FINAL_TEST_EVALUATION = True` only for the final selected run,
   then run the last evaluation cell once.

If the kernel was restarted after training, keep the same paths, set both
`FINAL_TEST_ONLY = True` and `CONFIRM_FINAL_TEST_EVALUATION = True`, then run
the notebook. It opens the completed run, skips training, reconstructs the fixed
test plan, and evaluates the existing validation-selected checkpoint.

Generated caches, checkpoints, JSON/NumPy reports, and figures stay in fresh,
ignored runtime directories and are not promoted into Git automatically.


## 1. Imports and runtime paths

The implementation stays importable cell-by-cell. Runtime paths remain
blank in the committed notebook; no cluster, PVC, namespace, credential,
or machine-specific path is embedded.


In [1]:
# Notebook-level utilities used by the orchestration cells below. Each embedded
# source module also retains its own public imports for direct traceability.
from dataclasses import replace
from pathlib import Path
from typing import Literal

import csv
import json
import os

import numpy as np
import torch


In [2]:
# Absolute runtime paths remain blank in Git. Set D4_ORQB_DATASETS_ROOT to the
# mounted datasets directory, or override any individual path variable.
DATASET_ID = "model_iv"
DATASETS_ROOT = os.environ.get("D4_ORQB_DATASETS_ROOT", "")

def _dataset_path(*parts):
    return os.path.join(DATASETS_ROOT, *parts) if DATASETS_ROOT.strip() else ""

DEVELOPMENT_ROOT = os.environ.get(
    "D4_ORQB_DEVELOPMENT_ROOT", _dataset_path('model_4', 'Model_IV', 'train')
)
VALIDATION_ROOT = os.environ.get(
    "D4_ORQB_VALIDATION_ROOT", _dataset_path('model_4', 'Model_IV', 'val')
)
# Model IV has no separate test dataset. Keep TEST_ROOT empty; a fixed
# class-stratified test holdout is carved from Model_IV/train.
TEST_ROOT = ""
CACHE_ROOT = os.environ.get("D4_ORQB_CACHE_ROOT", "")
OUTPUT_DIR = os.environ.get("D4_ORQB_OUTPUT_DIR", "")

# Notebook run family: source architecture and optimizer policy are unchanged;
# only the explicitly documented quantum epoch count is 50 instead of CLI default 40.
QUANTUM_EPOCHS = 50
# Carve 15% of Model_IV/train as the held-out test partition. The supplied
# Model_IV/val directory remains development validation.
TEST_FRACTION = 0.15
CONFIRM_FINAL_TEST_EVALUATION = os.environ.get(
    "D4_ORQB_CONFIRM_FINAL_TEST_EVALUATION", "0"
).strip().lower() in {"1", "true", "yes"}
# Set both flags True to reopen a completed OUTPUT_DIR after a kernel restart,
# reconstruct the fixed loaders, and run only the final held-out evaluation.
FINAL_TEST_ONLY = os.environ.get(
    "D4_ORQB_FINAL_TEST_ONLY", "0"
).strip().lower() in {"1", "true", "yes"}

# Model IV remains gated by the source audit. Enabling this records an explicitly
# research-only run when the audit is inconclusive; integrity failures and detected
# preprocessing signal loss can never be overridden.
ALLOW_INCONCLUSIVE_MODEL_IV_AUDIT = False

print({
    "dataset_id": DATASET_ID,
    "datasets_root_set": bool(DATASETS_ROOT.strip()),
    "development_root_set": bool(DEVELOPMENT_ROOT.strip()),
    "validation_root_set": bool(VALIDATION_ROOT.strip()),
    "test_root_set": bool(TEST_ROOT.strip()),
    "cache_root_set": bool(CACHE_ROOT.strip()),
    "output_dir_set": bool(OUTPUT_DIR.strip()),
    "quantum_epochs": QUANTUM_EPOCHS,
    "test_fraction": TEST_FRACTION,
    "final_test_only": FINAL_TEST_ONLY,
})


{'dataset_id': 'model_iv', 'datasets_root_set': False, 'development_root_set': True, 'validation_root_set': True, 'test_root_set': False, 'cache_root_set': True, 'output_dir_set': True, 'quantum_epochs': 50, 'test_fraction': 0.15, 'final_test_only': False}


## 2. Configuration contract

This cell is the complete `src/d4_orqb/config.py` implementation. It
validates dataset routing, stage policy, hyperparameters, and fresh runtime
locations. Test routing is deliberately a notebook-only extension because
the package CLI keeps official-test evaluation closed.


In [3]:
"""Configuration for the selected D4-ORQB training pipeline."""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Literal


DATASET_IDS = (
    "model_i",
    "model_ii",
    "model_iii",
    "model_iv",
    "model_v",
)
DatasetID = Literal[
    "model_i", "model_ii", "model_iii", "model_iv", "model_v"
]


@dataclass(slots=True)
class Config:
    """Runtime settings with no committed dataset location.

    ``stage="all"`` recreates the initialization used by the selected run:
    an 18-epoch classical context pretrain followed by a fresh 40-epoch
    quantum model initialized from the shared image backbone.
    """

    dataset_id: DatasetID = "model_i"
    development_root: str = ""
    validation_root: str = ""
    cache_root: str = ""
    output_dir: str = ""
    stage: Literal["all", "pretrain", "quantum", "audit"] = "all"
    backbone_checkpoint: str = ""
    freeze_backbone_during_quantum: bool = False
    allow_inconclusive_model_iv_audit: bool = False

    image_size: int = 96
    batch_size: int = 256
    workers: int = 4
    io_workers: int = 8
    val_fraction: float = 0.20
    split_seed: int = 42

    heads: int = 4
    reuploads: int = 2
    dropout: float = 0.10

    pretrain_epochs: int = 18
    pretrain_patience: int = 6
    pretrain_seed: int = 0
    pretrain_learning_rate: float = 4e-3
    pretrain_core_learning_rate: float = 6e-3

    quantum_epochs: int = 40
    quantum_patience: int = 41
    quantum_seed: int = 2
    encoder_learning_rate: float = 5e-4
    learning_rate: float = 3e-3
    core_learning_rate: float = 5e-3

    weight_decay: float = 1e-4
    label_smoothing: float = 0.02
    deterministic: bool = False

    def validate(self) -> None:
        if self.dataset_id not in DATASET_IDS:
            raise ValueError(
                f"Unknown dataset_id: {self.dataset_id}; "
                f"choose one of {DATASET_IDS}"
            )
        if not self.development_root.strip():
            raise ValueError(
                "Dataset path is empty. Set --development-root to the selected "
                "dataset's development directory on the training machine."
            )
        if self.dataset_id == "model_iv" and not self.validation_root.strip():
            raise ValueError(
                "Model IV requires --validation-root so its supplied "
                "development-validation split is preserved."
            )
        if self.validation_root.strip() and self.dataset_id != "model_iv":
            raise ValueError(
                "--validation-root is reserved for Model IV's supplied "
                "development-validation split. Official test evaluation is "
                "not supported by this training entry point."
            )
        if (
            self.validation_root.strip()
            and self.development_path == self.validation_path
        ):
            raise ValueError(
                "Development and validation roots must be different directories"
            )
        if self.stage != "audit" and not self.cache_root.strip():
            raise ValueError("Set --cache-root to a writable cache directory.")
        if not self.output_dir.strip():
            raise ValueError("Set --output-dir to a new run directory.")
        if self.stage not in ("all", "pretrain", "quantum", "audit"):
            raise ValueError(f"Unknown stage: {self.stage}")
        if self.stage == "audit" and self.dataset_id != "model_iv":
            raise ValueError("--stage audit is defined only for Model IV")
        if (
            self.allow_inconclusive_model_iv_audit
            and self.dataset_id != "model_iv"
        ):
            raise ValueError(
                "--allow-inconclusive-model-iv-audit applies only to Model IV"
            )
        if self.stage == "audit" and self.allow_inconclusive_model_iv_audit:
            raise ValueError(
                "The audit-only stage reports its real status and cannot be overridden"
            )
        if self.stage == "quantum" and self.backbone_checkpoint:
            checkpoint = Path(self.backbone_checkpoint).expanduser()
            if not checkpoint.is_file():
                raise FileNotFoundError(checkpoint)
        if self.freeze_backbone_during_quantum:
            if self.stage not in ("all", "quantum"):
                raise ValueError(
                    "--freeze-backbone-during-quantum is only valid when a "
                    "quantum stage is requested"
                )
            if self.stage == "quantum" and not self.backbone_checkpoint:
                raise ValueError(
                    "--freeze-backbone-during-quantum with --stage quantum "
                    "requires --backbone-checkpoint"
                )
        if self.image_size <= 0 or self.batch_size <= 0:
            raise ValueError("image_size and batch_size must be positive")
        if self.workers < 0 or self.io_workers <= 0:
            raise ValueError("workers must be nonnegative and io_workers positive")
        if not 0.0 < self.val_fraction < 1.0:
            raise ValueError("val_fraction must be between zero and one")
        if not 0.0 <= self.dropout < 1.0:
            raise ValueError("dropout must be in [0, 1)")
        if self.heads != 4 or self.reuploads != 2:
            raise ValueError(
                "The selected D4-ORQB circuit uses 4 heads and 2 reuploads"
            )
        for name in (
            "pretrain_epochs",
            "pretrain_patience",
            "quantum_epochs",
            "quantum_patience",
        ):
            if getattr(self, name) <= 0:
                raise ValueError(f"{name} must be positive")
        for name in (
            "pretrain_learning_rate",
            "pretrain_core_learning_rate",
            "encoder_learning_rate",
            "learning_rate",
            "core_learning_rate",
        ):
            if getattr(self, name) <= 0.0:
                raise ValueError(f"{name} must be positive")
        if self.weight_decay < 0.0:
            raise ValueError("weight_decay cannot be negative")
        if not 0.0 <= self.label_smoothing < 1.0:
            raise ValueError("label_smoothing must be in [0, 1)")

    @property
    def development_path(self) -> Path:
        return Path(self.development_root).expanduser().resolve()

    @property
    def validation_path(self) -> Path | None:
        if not self.validation_root.strip():
            return None
        return Path(self.validation_root).expanduser().resolve()

    @property
    def cache_path(self) -> Path:
        return Path(self.cache_root).expanduser().resolve()

    @property
    def cache_key(self) -> str:
        """Return a stable dataset- and resize-specific cache key."""

        return f"{self.dataset_id}_{self.image_size}"

    @property
    def output_path(self) -> Path:
        return Path(self.output_dir).expanduser().resolve()

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


## 3. Data extraction, caching, integrity audit, and base loaders

The full source data module follows. It recursively extracts the 2-D image
from supported NPY containers, cleans non-finite values, applies the source
normalization rule, hashes model-visible content, builds atomic resize
caches, and includes the complete Model-IV integrity/signal audit.


In [4]:
"""Leakage-safe dataset loading, resize caching, splitting, and loaders."""

from __future__ import annotations

import csv
import hashlib
import json
import os
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset



IMAGE_KEYS = ("image", "img", "x", "data", "array", "arr", "lens", "sample")
EXPECTED_CLASSES = ("axion", "cdm", "no_sub")
MODEL_IV_AUDIT_SEED = 20_260_715
MODEL_IV_AUDIT_TRAIN_CAP = 4_000
MODEL_IV_AUDIT_VALIDATION_CAP = 2_000
MODEL_IV_AUDIT_MIN_PER_CLASS = 500
MODEL_IV_AUDIT_BOOTSTRAPS = 1_000
MODEL_IV_AUDIT_PERMUTATIONS = 999

PASS_SIGNAL_DETECTED = "PASS_SIGNAL_DETECTED"
PREPROCESSING_SIGNAL_LOSS = "PREPROCESSING_SIGNAL_LOSS"
INCONCLUSIVE_NO_SIGNAL_DETECTED = "INCONCLUSIVE_NO_SIGNAL_DETECTED"
INTEGRITY_FAILED = "INTEGRITY_FAILED"

_AUDIT_ANNULI = 8
_AUDIT_POOL = 8


def extract_image_array(value) -> np.ndarray:
    """Extract only the two-dimensional image and discard scalar metadata."""

    if isinstance(value, np.ndarray):
        if value.dtype != object:
            return value
        if value.ndim == 0:
            return extract_image_array(value.item())
        for item in value.reshape(-1):
            candidate = extract_image_array(item)
            if np.asarray(candidate).ndim >= 2:
                return np.asarray(candidate)
        raise ValueError("Object array contains no image")
    if isinstance(value, dict):
        for key in IMAGE_KEYS:
            if key in value:
                candidate = extract_image_array(value[key])
                if np.asarray(candidate).ndim >= 2:
                    return np.asarray(candidate)
        for item in value.values():
            candidate = extract_image_array(item)
            if np.asarray(candidate).ndim >= 2:
                return np.asarray(candidate)
        raise ValueError("Dictionary contains no image")
    if isinstance(value, (list, tuple)):
        for item in value:
            candidate = extract_image_array(item)
            if np.asarray(candidate).ndim >= 2:
                return np.asarray(candidate)
        raise ValueError("Sequence contains no image")
    return np.asarray(value)


def load_model_visible_image(path: str | Path) -> Tuple[np.ndarray, str]:
    """Apply model-visible preprocessing and return a content digest."""

    raw = np.load(path, allow_pickle=True)
    image = np.asarray(extract_image_array(raw), dtype=np.float32).squeeze()
    if image.ndim != 2:
        raise ValueError(f"Expected a 2-D image in {path}, got {image.shape}")
    image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    maximum = float(image.max())
    if maximum > 1.0:
        image = image / maximum
    image = np.ascontiguousarray(image.astype("<f4", copy=False))
    return image, hashlib.sha256(image.tobytes()).hexdigest()


def list_samples(root: str | Path) -> Tuple[List[Tuple[str, int, str]], List[str]]:
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(root)
    classes = sorted(path.name for path in root.iterdir() if path.is_dir())
    if classes != list(EXPECTED_CLASSES):
        raise RuntimeError(
            "Class directories must be exactly "
            f"{list(EXPECTED_CLASSES)}; found {classes} under {root}"
        )
    samples: List[Tuple[str, int, str]] = []
    for label, class_name in enumerate(classes):
        class_dir = root / class_name
        class_samples = sorted(class_dir.glob("*.npy"))
        if not class_samples:
            raise RuntimeError(f"No .npy samples under {class_dir}")
        for path in class_samples:
            samples.append((str(path), label, str(path.relative_to(root))))
    return samples, classes


def _load_path(
    record: Tuple[str, int, str]
) -> Tuple[np.ndarray, int, str, str]:
    path, label, relative = record
    image, digest = load_model_visible_image(path)
    return image, label, relative, digest


def prepare_cache(
    source_root: str | Path,
    cache_dir: str | Path,
    image_size: int,
    device: torch.device,
    io_workers: int = 8,
    chunk_size: int = 384,
    storage_dtype=np.float16,
) -> Dict:
    """Create or reuse an atomic resize cache at the requested precision."""

    source_root = Path(source_root).resolve()
    cache_dir = Path(cache_dir)
    cache_dtype = np.dtype(storage_dtype)
    if cache_dtype not in (np.dtype(np.float16), np.dtype(np.float32)):
        raise ValueError(f"Unsupported cache dtype: {cache_dtype}")
    metadata_path = cache_dir / "metadata.json"
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        required = (
            cache_dir / "images.npy",
            cache_dir / "labels.npy",
            cache_dir / "manifest.csv",
        )
        if (
            metadata.get("complete")
            and metadata.get("image_size") == image_size
            and metadata.get("dtype") == cache_dtype.name
            and Path(metadata.get("source_root", "")) == source_root
            and metadata.get("classes") == list(EXPECTED_CLASSES)
            and all(path.exists() for path in required)
        ):
            print(
                f"CACHE_READY {cache_dir} samples={metadata['samples']}",
                flush=True,
            )
            return metadata

    cache_dir.mkdir(parents=True, exist_ok=True)
    samples, classes = list_samples(source_root)
    build_tag = f"building-{os.getpid()}"
    image_tmp = cache_dir / f"images-{build_tag}.npy"
    labels_tmp = cache_dir / f"labels-{build_tag}.npy"
    manifest_tmp = cache_dir / f"manifest-{build_tag}.csv"
    images_memmap = np.lib.format.open_memmap(
        image_tmp,
        mode="w+",
        dtype=cache_dtype,
        shape=(len(samples), image_size, image_size),
    )
    labels = np.empty(len(samples), dtype=np.int64)

    with manifest_tmp.open("w", newline="") as manifest_handle:
        writer = csv.writer(manifest_handle)
        writer.writerow(
            ("index", "relative_path", "class", "label", "sha256_visible")
        )
        with ThreadPoolExecutor(max_workers=io_workers) as pool:
            for start in range(0, len(samples), chunk_size):
                stop = min(start + chunk_size, len(samples))
                loaded = list(pool.map(_load_path, samples[start:stop]))
                batch = np.stack([item[0] for item in loaded], axis=0)
                tensor = torch.from_numpy(batch).unsqueeze(1).to(
                    device=device, dtype=torch.float32
                )
                resized = F.interpolate(
                    tensor,
                    size=(image_size, image_size),
                    mode="bilinear",
                    align_corners=False,
                    antialias=True,
                )
                output_dtype = (
                    torch.float32
                    if cache_dtype == np.dtype(np.float32)
                    else torch.float16
                )
                images_memmap[start:stop] = (
                    resized[:, 0].to(dtype=output_dtype).cpu().numpy()
                )
                for offset, (_, label, relative, digest) in enumerate(loaded):
                    index = start + offset
                    labels[index] = label
                    writer.writerow(
                        (index, relative, classes[label], label, digest)
                    )
                print(f"CACHE_PROGRESS {stop}/{len(samples)}", flush=True)

    images_memmap.flush()
    np.save(labels_tmp, labels)
    os.replace(image_tmp, cache_dir / "images.npy")
    os.replace(labels_tmp, cache_dir / "labels.npy")
    os.replace(manifest_tmp, cache_dir / "manifest.csv")
    metadata = {
        "complete": True,
        "source_root": str(source_root),
        "image_size": image_size,
        "samples": len(samples),
        "classes": classes,
        "class_counts": {
            classes[index]: int((labels == index).sum())
            for index in range(len(classes))
        },
        "normalization": "nonfinite cleanup; divide by max only when max > 1",
        "interpolation": "bilinear align_corners=False antialias=True",
        "dtype": cache_dtype.name,
    }
    metadata_tmp = cache_dir / f"metadata-{build_tag}.json"
    metadata_tmp.write_text(json.dumps(metadata, indent=2, sort_keys=True))
    os.replace(metadata_tmp, metadata_path)
    print(f"CACHE_COMPLETE {cache_dir}", flush=True)
    return metadata


def _visible_digests(cache_dir: str | Path) -> set[str]:
    """Return model-visible content hashes recorded by a completed cache."""

    manifest_path = Path(cache_dir) / "manifest.csv"
    with manifest_path.open(newline="") as manifest_handle:
        reader = csv.DictReader(manifest_handle)
        if reader.fieldnames is None or "sha256_visible" not in reader.fieldnames:
            raise RuntimeError(
                f"Cache manifest has no sha256_visible column: {manifest_path}"
            )
        return {row["sha256_visible"] for row in reader}


def _require_disjoint_visible_content(
    development_cache_dir: str | Path,
    validation_cache_dir: str | Path,
) -> None:
    """Reject model-visible samples shared by supplied train/validation roots."""

    overlap = _visible_digests(development_cache_dir).intersection(
        _visible_digests(validation_cache_dir)
    )
    if overlap:
        raise RuntimeError(
            "Supplied development and validation roots share "
            f"{len(overlap)} model-visible image digest(s); refusing a leaky run"
        )


@dataclass(slots=True)
class ModelIVAuditResult:
    """Outcome of the CPU-only, development-validation data gate."""

    status: str
    report_path: Path
    integrity_failures: List[str]
    probes: Dict[str, Dict[str, Any]]


def _schema_signature(value: Any) -> str:
    if isinstance(value, np.ndarray):
        return f"ndarray(shape={value.shape},dtype={value.dtype})"
    return type(value).__name__


def _inspect_audit_record(
    record: Tuple[str, int, str],
) -> Dict[str, Any]:
    path, label, relative = record
    try:
        value = np.load(path, allow_pickle=True)
        schema = _schema_signature(value)
        image = np.asarray(extract_image_array(value), dtype=np.float32).squeeze()
        if image.shape != (64, 64):
            return {
                "label": label,
                "relative": relative,
                "schema": schema,
                "failure": f"wrong extracted shape {image.shape}",
            }
        if not np.isfinite(image).all():
            return {
                "label": label,
                "relative": relative,
                "schema": schema,
                "failure": "nonfinite raw pixels",
            }
        minimum = float(image.min())
        maximum = float(image.max())
        if maximum == minimum:
            return {
                "label": label,
                "relative": relative,
                "schema": schema,
                "failure": "constant raw image",
            }
        visible = image.copy()
        if maximum > 1.0:
            visible /= maximum
        visible = np.ascontiguousarray(visible.astype("<f4", copy=False))
        return {
            "label": label,
            "relative": relative,
            "schema": schema,
            "failure": "",
            "digest": hashlib.sha256(visible.tobytes()).hexdigest(),
            "minimum": minimum,
            "maximum": maximum,
            "negative_fraction": float(np.mean(image < 0.0)),
        }
    except Exception as error:  # The report retains the path and short reason.
        return {
            "label": label,
            "relative": relative,
            "schema": "unreadable",
            "failure": f"{type(error).__name__}: {error}",
        }


def _balanced_audit_indices(
    labels: np.ndarray, per_class: int, seed: int
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    parts: List[np.ndarray] = []
    for label in range(len(EXPECTED_CLASSES)):
        choices = np.flatnonzero(labels == label)
        rng.shuffle(choices)
        parts.append(choices[: min(per_class, len(choices))])
    indices = np.concatenate(parts)
    rng.shuffle(indices)
    return indices


def _scan_audit_root(
    root: Path,
    split_name: str,
    io_workers: int,
) -> Tuple[
    List[Tuple[str, int, str]],
    np.ndarray,
    List[Dict[str, Any]],
    List[str],
]:
    samples, _ = list_samples(root)
    labels = np.asarray([record[1] for record in samples], dtype=np.int64)
    failures: List[str] = []
    for label, class_name in enumerate(EXPECTED_CLASSES):
        count = int(np.sum(labels == label))
        if count < MODEL_IV_AUDIT_MIN_PER_CLASS:
            failures.append(
                f"{split_name}/{class_name} has {count} samples; "
                f"at least {MODEL_IV_AUDIT_MIN_PER_CLASS} are required"
            )

    rows: List[Dict[str, Any]] = []
    failure_counts: Dict[str, int] = {}
    failure_examples: Dict[str, str] = {}
    workers = max(1, min(io_workers, _AUDIT_POOL))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for index, row in enumerate(pool.map(_inspect_audit_record, samples), 1):
            rows.append(row)
            reason = str(row["failure"])
            if reason:
                failure_counts[reason] = failure_counts.get(reason, 0) + 1
                failure_examples.setdefault(reason, str(row["relative"]))
            if index % 5_000 == 0 or index == len(samples):
                print(
                    f"MODEL_IV_AUDIT_SCAN {split_name} {index}/{len(samples)}",
                    flush=True,
                )
    for reason, count in sorted(failure_counts.items()):
        failures.append(
            f"{split_name}: {count} file(s) failed {reason}; "
            f"example={failure_examples[reason]}"
        )
    return samples, labels, rows, failures


def _digest_integrity(
    development_rows: List[Dict[str, Any]],
    validation_rows: List[Dict[str, Any]],
) -> Tuple[List[str], Dict[str, int]]:
    failures: List[str] = []

    def digest_map(rows: List[Dict[str, Any]]) -> Dict[str, List[int]]:
        output: Dict[str, List[int]] = {}
        for row in rows:
            digest = row.get("digest")
            if digest:
                output.setdefault(str(digest), []).append(int(row["label"]))
        return output

    development = digest_map(development_rows)
    validation = digest_map(validation_rows)
    cross_label_development = sum(
        len(set(labels)) > 1 for labels in development.values()
    )
    cross_label_validation = sum(
        len(set(labels)) > 1 for labels in validation.values()
    )
    overlap = len(set(development).intersection(validation))
    if cross_label_development:
        failures.append(
            "development contains "
            f"{cross_label_development} model-visible digest(s) across labels"
        )
    if cross_label_validation:
        failures.append(
            "validation contains "
            f"{cross_label_validation} model-visible digest(s) across labels"
        )
    if overlap:
        failures.append(
            f"development and validation share {overlap} model-visible digest(s)"
        )
    same_label_duplicates = 0
    for mapping in (development, validation):
        for labels in mapping.values():
            if len(set(labels)) == 1 and len(labels) > 1:
                same_label_duplicates += len(labels) - 1
    return failures, {
        "cross_label_development": cross_label_development,
        "cross_label_validation": cross_label_validation,
        "development_validation_overlap": overlap,
        "same_label_duplicate_copies": same_label_duplicates,
    }


def _load_raw_audit_image(record: Tuple[str, int, str]) -> np.ndarray:
    value = np.load(record[0], allow_pickle=True)
    return np.asarray(extract_image_array(value), dtype=np.float32).squeeze()


def _load_audit_subset(
    samples: List[Tuple[str, int, str]],
    indices: np.ndarray,
    io_workers: int,
) -> np.ndarray:
    records = [samples[int(index)] for index in indices]
    workers = max(1, min(io_workers, _AUDIT_POOL))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        images = list(pool.map(_load_raw_audit_image, records))
    return np.stack(images).astype(np.float32, copy=False)


def _model_visible_chunk(images: np.ndarray, image_size: int) -> np.ndarray:
    visible = np.asarray(images, dtype=np.float32).copy()
    maxima = visible.max(axis=(1, 2), keepdims=True)
    divide = maxima[:, 0, 0] > 1.0
    visible[divide] /= maxima[divide]
    if visible.shape[-2:] != (image_size, image_size):
        tensor = F.interpolate(
            torch.from_numpy(visible).unsqueeze(1),
            size=(image_size, image_size),
            mode="bilinear",
            align_corners=False,
            antialias=True,
        )
        visible = tensor[:, 0].numpy()
    return visible


def _d4_feature_chunk(images: np.ndarray) -> np.ndarray:
    """Extract fixed D4-invariant radial and morphology summaries."""

    x = np.asarray(images, dtype=np.float32)
    count, height, width = x.shape
    yy, xx = np.mgrid[-1:1:complex(height), -1:1:complex(width)]
    radius = np.sqrt(xx * xx + yy * yy)
    theta = np.arctan2(yy, xx)
    edges = np.linspace(0.0, np.sqrt(2.0) + 1e-6, _AUDIT_ANNULI + 1)

    padded = np.pad(x, ((0, 0), (1, 1), (1, 1)), mode="reflect")
    gx = 0.5 * (padded[:, 1:-1, 2:] - padded[:, 1:-1, :-2])
    gy = 0.5 * (padded[:, 2:, 1:-1] - padded[:, :-2, 1:-1])
    gradient = np.hypot(gx, gy)
    laplacian = (
        padded[:, 1:-1, 2:]
        + padded[:, 1:-1, :-2]
        + padded[:, 2:, 1:-1]
        + padded[:, :-2, 1:-1]
        - 4.0 * x
    )
    smooth = sum(
        padded[:, dy : dy + height, dx : dx + width]
        for dy in range(3)
        for dx in range(3)
    ) / 9.0
    highpass = x - smooth

    features: List[np.ndarray] = []
    weights_all = np.abs(x)
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (radius >= lower) & (radius < upper)
        for channel in (x, gradient, laplacian, highpass):
            features.append(np.sqrt(np.mean(channel[:, mask] ** 2, axis=1)))
        weights = weights_all[:, mask]
        denominator = np.maximum(weights.sum(axis=1), 1e-8)
        ring_theta = theta[mask]
        for mode in range(1, 5):
            moment = (
                weights * np.exp(1j * mode * ring_theta)[None, :]
            ).sum(axis=1) / denominator
            features.append(np.abs(moment))

    # Use the full Fourier plane: an unweighted rFFT half-plane gives the
    # Nyquist/DC boundary different multiplicities after a 90-degree rotation.
    power = np.abs(np.fft.fft2(x, axes=(-2, -1))) ** 2
    frequency_y = np.fft.fftfreq(height)[:, None]
    frequency_x = np.fft.fftfreq(width)[None, :]
    frequency_radius = np.sqrt(frequency_x**2 + frequency_y**2)
    frequency_edges = np.linspace(
        0.0, float(frequency_radius.max()) + 1e-8, _AUDIT_ANNULI + 1
    )
    for lower, upper in zip(frequency_edges[:-1], frequency_edges[1:]):
        mask = (frequency_radius >= lower) & (frequency_radius < upper)
        features.append(np.log1p(power[:, mask].mean(axis=1)))

    orbit = []
    for rotation in range(4):
        rotated = np.rot90(x, rotation, axes=(-2, -1))
        orbit.extend((rotated, np.flip(rotated, axis=-1)))
    invariant_image = np.mean(orbit, axis=0)
    if height % 8 == 0 and width % 8 == 0:
        coarse = invariant_image.reshape(
            count, 8, height // 8, 8, width // 8
        ).mean(axis=(2, 4))
    else:
        coarse = F.adaptive_avg_pool2d(
            torch.from_numpy(invariant_image).unsqueeze(1), (8, 8)
        )[:, 0].numpy()
    features.extend(coarse.reshape(count, -1).T)
    features.extend(
        (
            x.mean(axis=(1, 2)),
            x.std(axis=(1, 2)),
            x.min(axis=(1, 2)),
            x.max(axis=(1, 2)),
            np.mean(np.abs(x - invariant_image), axis=(1, 2)),
        )
    )
    output = np.stack(features, axis=1).astype(np.float32)
    return np.nan_to_num(output, nan=0.0, posinf=1e20, neginf=-1e20)


def _audit_features(
    images: np.ndarray,
    model_visible: bool,
    image_size: int,
    chunk_size: int = 256,
) -> np.ndarray:
    chunks: List[np.ndarray] = []
    for start in range(0, len(images), chunk_size):
        chunk = images[start : start + chunk_size]
        if model_visible:
            chunk = _model_visible_chunk(chunk, image_size)
        chunks.append(_d4_feature_chunk(chunk))
    return np.concatenate(chunks, axis=0)


def _fit_fixed_ridge(
    train_features: np.ndarray,
    train_labels: np.ndarray,
    validation_features: np.ndarray,
) -> np.ndarray:
    train = np.asarray(train_features, dtype=np.float64)
    validation = np.asarray(validation_features, dtype=np.float64)
    mean = train.mean(axis=0)
    scale = train.std(axis=0)
    scale[scale < 1e-8] = 1.0
    train = np.clip((train - mean) / scale, -30.0, 30.0)
    validation = np.clip((validation - mean) / scale, -30.0, 30.0)
    train = np.concatenate((np.ones((len(train), 1)), train), axis=1)
    validation = np.concatenate(
        (np.ones((len(validation), 1)), validation), axis=1
    )
    targets = np.eye(len(EXPECTED_CLASSES), dtype=np.float64)[train_labels]
    penalty = np.eye(train.shape[1], dtype=np.float64)
    penalty[0, 0] = 0.0
    weights = np.linalg.solve(
        train.T @ train + penalty,
        train.T @ targets,
    )
    return validation @ weights


def _average_ranks(values: np.ndarray) -> np.ndarray:
    order = np.argsort(values, kind="mergesort")
    sorted_values = values[order]
    ranks = np.empty(len(values), dtype=np.float64)
    start = 0
    while start < len(values):
        stop = start + 1
        while stop < len(values) and sorted_values[stop] == sorted_values[start]:
            stop += 1
        ranks[order[start:stop]] = 0.5 * (start + stop + 1)
        start = stop
    return ranks


def _binary_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    positive = np.asarray(labels, dtype=bool)
    positive_count = int(positive.sum())
    negative_count = len(positive) - positive_count
    if not positive_count or not negative_count:
        return float("nan")
    ranks = _average_ranks(np.asarray(scores, dtype=np.float64))
    numerator = ranks[positive].sum() - positive_count * (positive_count + 1) / 2
    return float(numerator / (positive_count * negative_count))


def _score_metrics(labels: np.ndarray, scores: np.ndarray) -> Dict[str, Any]:
    prediction = np.asarray(scores).argmax(axis=1)
    confusion = np.zeros((len(EXPECTED_CLASSES), len(EXPECTED_CLASSES)), dtype=int)
    for truth, predicted in zip(labels, prediction):
        confusion[int(truth), int(predicted)] += 1
    recalls = np.divide(
        np.diag(confusion),
        confusion.sum(axis=1),
        out=np.zeros(len(EXPECTED_CLASSES), dtype=float),
        where=confusion.sum(axis=1) != 0,
    )
    f1_values = []
    for label in range(len(EXPECTED_CLASSES)):
        true_positive = confusion[label, label]
        false_positive = confusion[:, label].sum() - true_positive
        false_negative = confusion[label, :].sum() - true_positive
        denominator = 2 * true_positive + false_positive + false_negative
        f1_values.append(0.0 if denominator == 0 else 2 * true_positive / denominator)
    per_class_auc = [
        _binary_auc(labels == label, scores[:, label])
        for label in range(len(EXPECTED_CLASSES))
    ]
    return {
        "accuracy": float(np.mean(prediction == labels)),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1_values)),
        "macro_auc_ovr": float(np.mean(per_class_auc)),
        "per_class_auc": per_class_auc,
        "confusion_matrix": confusion.tolist(),
    }


def _bootstrap_probe(
    labels: np.ndarray,
    scores: np.ndarray,
    seed: int,
) -> Dict[str, Any]:
    rng = np.random.default_rng(seed)
    class_indices = [
        np.flatnonzero(labels == label) for label in range(len(EXPECTED_CLASSES))
    ]
    balanced_accuracy = np.empty(MODEL_IV_AUDIT_BOOTSTRAPS)
    macro_auc = np.empty(MODEL_IV_AUDIT_BOOTSTRAPS)
    per_class_auc = np.empty(
        (MODEL_IV_AUDIT_BOOTSTRAPS, len(EXPECTED_CLASSES))
    )
    for iteration in range(MODEL_IV_AUDIT_BOOTSTRAPS):
        sampled = np.concatenate(
            [rng.choice(indices, len(indices), replace=True) for indices in class_indices]
        )
        metrics = _score_metrics(labels[sampled], scores[sampled])
        balanced_accuracy[iteration] = metrics["balanced_accuracy"]
        macro_auc[iteration] = metrics["macro_auc_ovr"]
        per_class_auc[iteration] = metrics["per_class_auc"]
    return {
        "balanced_accuracy_ci95": np.quantile(
            balanced_accuracy, (0.025, 0.975)
        ).tolist(),
        "macro_auc_ovr_ci95": np.quantile(
            macro_auc, (0.025, 0.975)
        ).tolist(),
        "per_class_auc_ci95": np.quantile(
            per_class_auc, (0.025, 0.975), axis=0
        ).T.tolist(),
        "bootstrap_repetitions": MODEL_IV_AUDIT_BOOTSTRAPS,
    }


def _permutation_max_t(
    labels: np.ndarray,
    scores_by_view: Dict[str, np.ndarray],
    seed: int,
) -> Dict[str, float]:
    rng = np.random.default_rng(seed)
    observed = {
        name: _score_metrics(labels, scores)["macro_auc_ovr"]
        for name, scores in scores_by_view.items()
    }
    ranks = {
        name: np.stack(
            [_average_ranks(scores[:, label]) for label in range(len(EXPECTED_CLASSES))],
            axis=1,
        )
        for name, scores in scores_by_view.items()
    }
    positive_counts = np.asarray(
        [np.sum(labels == label) for label in range(len(EXPECTED_CLASSES))]
    )
    negative_counts = len(labels) - positive_counts
    max_null = np.empty(MODEL_IV_AUDIT_PERMUTATIONS)
    for iteration in range(MODEL_IV_AUDIT_PERMUTATIONS):
        permuted = rng.permutation(labels)
        view_statistics = []
        for view_ranks in ranks.values():
            aucs = []
            for label in range(len(EXPECTED_CLASSES)):
                positive = permuted == label
                numerator = view_ranks[positive, label].sum() - (
                    positive_counts[label] * (positive_counts[label] + 1) / 2
                )
                aucs.append(
                    numerator / (positive_counts[label] * negative_counts[label])
                )
            view_statistics.append(float(np.mean(aucs)))
        max_null[iteration] = max(view_statistics)
    return {
        name: float(
            (1 + np.sum(max_null >= statistic))
            / (MODEL_IV_AUDIT_PERMUTATIONS + 1)
        )
        for name, statistic in observed.items()
    }


def _probe_passes(probe: Dict[str, Any]) -> bool:
    return bool(
        probe["balanced_accuracy"] >= 0.40
        and probe["macro_auc_ovr"] >= 0.55
        and probe["macro_auc_ovr_ci95"][0] >= 0.52
        and all(interval[0] > 0.50 for interval in probe["per_class_auc_ci95"])
        and probe["permutation_max_t_p"] <= 0.01
    )


def _schema_summary(
    rows: List[Dict[str, Any]], labels: np.ndarray
) -> Dict[str, Dict[str, int]]:
    summary: Dict[str, Dict[str, int]] = {}
    for label, class_name in enumerate(EXPECTED_CLASSES):
        counts: Dict[str, int] = {}
        for row in rows:
            if int(row["label"]) == label:
                schema = str(row["schema"])
                counts[schema] = counts.get(schema, 0) + 1
        summary[class_name] = counts
    return summary


def _write_audit_report(
    path: Path,
    status: str,
    counts: Dict[str, Dict[str, int]],
    schemas: Dict[str, Dict[str, Dict[str, int]]],
    integrity: Dict[str, int],
    failures: List[str],
    probes: Dict[str, Dict[str, Any]],
) -> None:
    lines = [
        "# Model IV dataset audit",
        "",
        f"- Status: `{status}`",
        "- Evaluation scope: supplied development-validation only",
        "- Official test evaluated: `false`",
        f"- Fixed seed: `{MODEL_IV_AUDIT_SEED}`",
        "",
        "This is an operational preflight gate. Failure to detect signal with "
        "this fixed probe does not prove that the Bayes-optimal signal is zero.",
        "",
        "## Integrity",
        "",
        "| Split | axion | cdm | no_sub |",
        "| --- | ---: | ---: | ---: |",
        "| development | {axion} | {cdm} | {no_sub} |".format(
            **{
                name: counts.get("development", {}).get(name, 0)
                for name in EXPECTED_CLASSES
            }
        ),
        "| validation | {axion} | {cdm} | {no_sub} |".format(
            **{
                name: counts.get("validation", {}).get(name, 0)
                for name in EXPECTED_CLASSES
            }
        ),
        "",
        f"- Cross-label development digests: `{integrity.get('cross_label_development', 0)}`",
        f"- Cross-label validation digests: `{integrity.get('cross_label_validation', 0)}`",
        f"- Development/validation digest overlap: `{integrity.get('development_validation_overlap', 0)}`",
        f"- Same-label duplicate copies (reported, not by itself fatal): `{integrity.get('same_label_duplicate_copies', 0)}`",
        "",
    ]
    if failures:
        lines.extend(("### Integrity failures", ""))
        lines.extend(f"- {failure}" for failure in failures)
        lines.append("")
    lines.extend(
        (
            "### Raw serialization schemas (warning-only provenance evidence)",
            "",
        )
    )
    for split_name, split_schemas in schemas.items():
        lines.append(f"- **{split_name}**")
        for class_name, class_schemas in split_schemas.items():
            rendered = ", ".join(
                f"`{schema}`: {count}" for schema, count in sorted(class_schemas.items())
            )
            lines.append(f"  - {class_name}: {rendered}")
    lines.extend(
        (
            "",
            "Schema differences are never used as classifier features and are "
            "not treated as proof of pixel corruption.",
            "",
            "## Frozen signal probe",
            "",
            "The probe uses only pixels: D4-invariant annular intensity, "
            "gradient, Laplacian and high-pass energies; angular multipole "
            "magnitudes; radial Fourier power; and an eight-view symmetrized "
            "coarse image. A fixed one-vs-rest ridge model is standardized on "
            "development data only and scored once on supplied validation.",
            "",
        )
    )
    if probes:
        lines.extend(
            (
                "| View | N train | N validation | Accuracy | Balanced accuracy (95% CI) | Macro OVR AUC (95% CI) | max-T p | Pass |",
                "| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |",
            )
        )
        for name in ("raw", "model_visible"):
            if name not in probes:
                continue
            probe = probes[name]
            ba_ci = probe["balanced_accuracy_ci95"]
            auc_ci = probe["macro_auc_ovr_ci95"]
            lines.append(
                f"| {name} | {probe['n_train']} | {probe['n_validation']} | "
                f"{probe['accuracy']:.5f} | {probe['balanced_accuracy']:.5f} "
                f"[{ba_ci[0]:.5f}, {ba_ci[1]:.5f}] | "
                f"{probe['macro_auc_ovr']:.5f} "
                f"[{auc_ci[0]:.5f}, {auc_ci[1]:.5f}] | "
                f"{probe['permutation_max_t_p']:.4f} | "
                f"{'yes' if probe['passes'] else 'no'} |"
            )
        lines.extend(("", "Per-class OVR AUC confidence intervals:", ""))
        for name, probe in probes.items():
            rendered = ", ".join(
                f"{class_name}={probe['per_class_auc'][index]:.5f} "
                f"[{probe['per_class_auc_ci95'][index][0]:.5f}, "
                f"{probe['per_class_auc_ci95'][index][1]:.5f}]"
                for index, class_name in enumerate(EXPECTED_CLASSES)
            )
            lines.append(f"- **{name}:** {rendered}")
        lines.extend(
            (
                "",
                "A view passes only when balanced accuracy is at least 0.40, "
                "macro AUC is at least 0.55, its bootstrap lower bound is at "
                "least 0.52, every class-AUC lower bound exceeds 0.50, and the "
                "two-view max-T permutation p-value is at most 0.01.",
                "",
                "The archive has no pair/source IDs, so confidence intervals "
                "use a sample-level stratified bootstrap and can be optimistic "
                "under source reuse. A repaired release must use grouped "
                "source/pair inference.",
                "",
            )
        )
    if status == PREPROCESSING_SIGNAL_LOSS:
        lines.append(
            "Raw pixels pass while the exact model-visible view does not; "
            "training is blocked until preprocessing preserves the signal."
        )
    elif status == INCONCLUSIVE_NO_SIGNAL_DETECTED:
        lines.append(
            "Neither fixed view demonstrates held-out signal. The archive is "
            "quarantined before GPU training; this is not a proof of no signal."
        )
    elif status == INTEGRITY_FAILED:
        lines.append("Definite integrity failures block any signal interpretation.")
    elif status == PASS_SIGNAL_DETECTED:
        lines.append("The model-visible development-validation signal gate passes.")
    lines.append("")
    path.write_text("\n".join(lines))


def run_model_iv_audit(config: Config, output_root: str | Path) -> ModelIVAuditResult:
    """Run the fixed Model-IV integrity and signal audit without CUDA."""

    if config.dataset_id != "model_iv" or config.validation_path is None:
        raise ValueError("The Model-IV audit requires Model IV and supplied validation")
    report_path = Path(output_root) / "dataset_audit.md"
    failures: List[str] = []
    probes: Dict[str, Dict[str, Any]] = {}
    counts: Dict[str, Dict[str, int]] = {}
    schemas: Dict[str, Dict[str, Dict[str, int]]] = {}
    integrity: Dict[str, int] = {}
    try:
        development_samples, development_labels, development_rows, dev_failures = (
            _scan_audit_root(
                config.development_path, "development", config.io_workers
            )
        )
        validation_samples, validation_labels, validation_rows, val_failures = (
            _scan_audit_root(
                config.validation_path, "validation", config.io_workers
            )
        )
        failures.extend(dev_failures)
        failures.extend(val_failures)
        digest_failures, integrity = _digest_integrity(
            development_rows, validation_rows
        )
        failures.extend(digest_failures)
        counts = {
            "development": {
                class_name: int(np.sum(development_labels == label))
                for label, class_name in enumerate(EXPECTED_CLASSES)
            },
            "validation": {
                class_name: int(np.sum(validation_labels == label))
                for label, class_name in enumerate(EXPECTED_CLASSES)
            },
        }
        schemas = {
            "development": _schema_summary(development_rows, development_labels),
            "validation": _schema_summary(validation_rows, validation_labels),
        }
    except Exception as error:
        failures.append(f"dataset discovery failed: {type(error).__name__}: {error}")
        _write_audit_report(
            report_path,
            INTEGRITY_FAILED,
            counts,
            schemas,
            integrity,
            failures,
            probes,
        )
        return ModelIVAuditResult(
            INTEGRITY_FAILED, report_path, failures, probes
        )

    if failures:
        _write_audit_report(
            report_path,
            INTEGRITY_FAILED,
            counts,
            schemas,
            integrity,
            failures,
            probes,
        )
        return ModelIVAuditResult(
            INTEGRITY_FAILED, report_path, failures, probes
        )

    development_indices = _balanced_audit_indices(
        development_labels,
        MODEL_IV_AUDIT_TRAIN_CAP,
        MODEL_IV_AUDIT_SEED + 1,
    )
    validation_indices = _balanced_audit_indices(
        validation_labels,
        MODEL_IV_AUDIT_VALIDATION_CAP,
        MODEL_IV_AUDIT_SEED + 2,
    )
    development_images = _load_audit_subset(
        development_samples, development_indices, config.io_workers
    )
    validation_images = _load_audit_subset(
        validation_samples, validation_indices, config.io_workers
    )
    train_labels = development_labels[development_indices]
    heldout_labels = validation_labels[validation_indices]

    scores_by_view: Dict[str, np.ndarray] = {}
    for name, model_visible in (("raw", False), ("model_visible", True)):
        train_features = _audit_features(
            development_images, model_visible, config.image_size
        )
        validation_features = _audit_features(
            validation_images, model_visible, config.image_size
        )
        scores_by_view[name] = _fit_fixed_ridge(
            train_features, train_labels, validation_features
        )
        print(
            f"MODEL_IV_AUDIT_PROBE_FEATURES view={name} "
            f"dimension={train_features.shape[1]}",
            flush=True,
        )

    corrected_p = _permutation_max_t(
        heldout_labels, scores_by_view, MODEL_IV_AUDIT_SEED + 3
    )
    for offset, (name, scores) in enumerate(scores_by_view.items()):
        probe = _score_metrics(heldout_labels, scores)
        probe.update(
            _bootstrap_probe(
                heldout_labels,
                scores,
                MODEL_IV_AUDIT_SEED + 100 + offset,
            )
        )
        probe.update(
            {
                "n_train": int(len(train_labels)),
                "n_validation": int(len(heldout_labels)),
                "permutation_max_t_p": corrected_p[name],
            }
        )
        probe["passes"] = _probe_passes(probe)
        probes[name] = probe

    if probes["model_visible"]["passes"]:
        status = PASS_SIGNAL_DETECTED
    elif probes["raw"]["passes"]:
        status = PREPROCESSING_SIGNAL_LOSS
    else:
        status = INCONCLUSIVE_NO_SIGNAL_DETECTED
    _write_audit_report(
        report_path,
        status,
        counts,
        schemas,
        integrity,
        failures,
        probes,
    )
    return ModelIVAuditResult(status, report_path, failures, probes)


def fixed_stratified_split(
    labels: np.ndarray,
    split_path: str | Path,
    val_fraction: float = 0.20,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create or verify the fixed class-stratified development split."""

    split_path = Path(split_path)
    if split_path.exists():
        saved = np.load(split_path)
        train, validation = saved["train"], saved["val"]
        if "seed" in saved and int(saved["seed"]) != seed:
            raise RuntimeError(f"Cached split seed mismatch in {split_path}")
        if "val_fraction" in saved and not np.isclose(
            float(saved["val_fraction"]),
            val_fraction,
            rtol=0.0,
            atol=1e-12,
        ):
            raise RuntimeError(
                f"Cached validation fraction mismatch in {split_path}"
            )
    else:
        rng = np.random.default_rng(seed)
        train_parts, validation_parts = [], []
        for label in sorted(np.unique(labels).tolist()):
            indices = np.flatnonzero(labels == label)
            rng.shuffle(indices)
            validation_count = int(round(len(indices) * val_fraction))
            validation_parts.append(indices[:validation_count])
            train_parts.append(indices[validation_count:])
        train = np.concatenate(train_parts)
        validation = np.concatenate(validation_parts)
        rng.shuffle(train)
        rng.shuffle(validation)
        split_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = split_path.with_name(
            f"{split_path.stem}-building-{os.getpid()}.npz"
        )
        np.savez(
            temporary,
            train=train,
            val=validation,
            seed=seed,
            val_fraction=val_fraction,
        )
        os.replace(temporary, split_path)

    train = np.asarray(train, dtype=np.int64)
    validation = np.asarray(validation, dtype=np.int64)
    if train.ndim != 1 or validation.ndim != 1:
        raise RuntimeError("Split indices must be one-dimensional")
    if len(np.unique(train)) != len(train) or len(np.unique(validation)) != len(
        validation
    ):
        raise RuntimeError("Split contains duplicate indices")
    if np.intersect1d(train, validation, assume_unique=True).size:
        raise RuntimeError("Training and validation indices overlap")
    if len(train) + len(validation) != len(labels):
        raise RuntimeError("Split does not cover the development dataset")
    combined = np.sort(np.concatenate((train, validation)))
    if not np.array_equal(combined, np.arange(len(labels))):
        raise RuntimeError("Split coverage does not match development indices")
    return train, validation


class CachedNPYDataset(Dataset):
    def __init__(
        self, cache_dir: str | Path, indices: Optional[np.ndarray] = None
    ) -> None:
        self.cache_dir = Path(cache_dir)
        self.images_path = self.cache_dir / "images.npy"
        self.labels = np.load(self.cache_dir / "labels.npy")
        self.indices = (
            np.arange(len(self.labels), dtype=np.int64)
            if indices is None
            else np.asarray(indices, dtype=np.int64)
        )
        self._images = None

    @property
    def images(self):
        if self._images is None:
            self._images = np.load(self.images_path, mmap_mode="r")
        return self._images

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int):
        index = int(self.indices[item])
        image = np.array(self.images[index], copy=True)
        return (
            torch.from_numpy(image).unsqueeze(0),
            int(self.labels[index]),
            index,
        )

    def __getstate__(self):
        state = dict(self.__dict__)
        state["_images"] = None
        return state


def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    workers: int,
    seed: int,
) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    options = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=True,
        drop_last=False,
        generator=generator,
    )
    if workers:
        options.update(persistent_workers=True, prefetch_factor=3)
    return DataLoader(**options)


@dataclass(slots=True)
class LoaderBundle:
    train: DataLoader
    validation: DataLoader
    class_names: List[str]
    train_indices: np.ndarray
    validation_indices: np.ndarray
    metadata: Dict


def build_loaders(
    config: Config,
    seed: int,
    device: torch.device,
) -> LoaderBundle:
    """Build loaders with an internal or supplied development-validation split."""

    development_cache_dir = config.cache_path / config.cache_key
    cache_storage_dtype = (
        np.float32 if config.dataset_id == "model_iv" else np.float16
    )
    development_metadata = prepare_cache(
        config.development_path,
        development_cache_dir,
        config.image_size,
        device,
        io_workers=config.io_workers,
        storage_dtype=cache_storage_dtype,
    )
    development_labels = np.load(development_cache_dir / "labels.npy")
    validation_path = config.validation_path
    metadata = dict(development_metadata)

    if validation_path is None:
        train_indices, validation_indices = fixed_stratified_split(
            development_labels,
            config.output_path / "split_indices.npz",
            val_fraction=config.val_fraction,
            seed=config.split_seed,
        )
        train_dataset = CachedNPYDataset(
            development_cache_dir, train_indices
        )
        validation_dataset = CachedNPYDataset(
            development_cache_dir, validation_indices
        )
        metadata["validation_mode"] = "fixed_stratified_development_split"
    else:
        validation_cache_dir = (
            config.cache_path / f"{config.cache_key}_validation"
        )
        validation_metadata = prepare_cache(
            validation_path,
            validation_cache_dir,
            config.image_size,
            device,
            io_workers=config.io_workers,
            storage_dtype=cache_storage_dtype,
        )
        if development_metadata["classes"] != validation_metadata["classes"]:
            raise RuntimeError(
                "Development and validation class mappings do not match: "
                f"{development_metadata['classes']} != "
                f"{validation_metadata['classes']}"
            )
        _require_disjoint_visible_content(
            development_cache_dir, validation_cache_dir
        )
        validation_labels = np.load(validation_cache_dir / "labels.npy")
        train_indices = np.arange(len(development_labels), dtype=np.int64)
        validation_indices = np.arange(
            len(validation_labels), dtype=np.int64
        )
        train_dataset = CachedNPYDataset(
            development_cache_dir, train_indices
        )
        validation_dataset = CachedNPYDataset(
            validation_cache_dir, validation_indices
        )
        metadata["validation_mode"] = "supplied_development_validation"
        metadata["visible_content_overlap"] = 0
        metadata["validation"] = validation_metadata

    train_loader = make_loader(
        train_dataset,
        config.batch_size,
        shuffle=True,
        workers=config.workers,
        seed=seed,
    )
    metadata["training_sampler"] = {"kind": "random_batches"}

    return LoaderBundle(
        train=train_loader,
        validation=make_loader(
            validation_dataset,
            config.batch_size,
            shuffle=False,
            workers=config.workers,
            seed=seed + 10_000,
        ),
        class_names=list(development_metadata["classes"]),
        train_indices=train_indices,
        validation_indices=validation_indices,
        metadata=metadata,
    )


### Notebook split and final-test extension

The source engine intentionally has only train/validation loaders. The
following transparent extension implements the `main` notebook split
contract without changing preprocessing or model behavior. It creates one
partition before datasets/loaders, persists it, checks index coverage and
content digests, fingerprints the ordered sample identities used by every
index, and returns a test *plan* that training cannot consume.


In [5]:
@dataclass(slots=True)
class HeldoutTestPlan:
    """Description of a test set that is never used by the training engine."""

    kind: Literal["official", "carved"]
    class_names: List[str]
    cache_dir: Path
    indices: np.ndarray | None = None
    source_root: str = ""
    description: str = ""


def sample_identity_fingerprint(
    records: List[Tuple[str, int, str]],
) -> str:
    """Hash the ordered label/relative-path identity used by persisted indices."""

    digest = hashlib.sha256()
    for _, label, relative in records:
        digest.update(f"{label}\0{relative}\n".encode())
    return digest.hexdigest()


def cache_identity_fingerprint(cache_dir: str | Path) -> str:
    """Hash cached manifest identities in their exact numeric index order."""

    with (Path(cache_dir) / "manifest.csv").open(newline="") as handle:
        rows = sorted(csv.DictReader(handle), key=lambda row: int(row["index"]))
    records = [
        ("", int(row["label"]), row["relative_path"])
        for row in rows
    ]
    return sample_identity_fingerprint(records)


def fixed_stratified_train_validation_test_split(
    labels: np.ndarray,
    split_path: str | Path,
    val_fraction: float,
    test_fraction: float,
    seed: int,
    sample_fingerprint: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Create or verify one fixed, class-stratified three-way partition.

    Test indices are selected first, validation indices second, and training gets
    the remainder. Fractions always refer to the original per-class population.
    This prevents the common error of taking 15% from an already reduced dataset.
    """

    if not 0.0 <= val_fraction < 1.0:
        raise ValueError("val_fraction must be in [0, 1)")
    if not 0.0 <= test_fraction < 1.0:
        raise ValueError("test_fraction must be in [0, 1)")
    if val_fraction + test_fraction >= 1.0:
        raise ValueError("validation + test fractions must leave training data")
    if not sample_fingerprint:
        raise ValueError("sample_fingerprint is required for split identity")

    labels = np.asarray(labels, dtype=np.int64)
    split_path = Path(split_path)
    if split_path.exists():
        saved = np.load(split_path)
        train = saved["train"]
        validation = saved["val"]
        test = saved["test"]
        expected = {
            "seed": seed,
            "val_fraction": val_fraction,
            "test_fraction": test_fraction,
            "sample_fingerprint": sample_fingerprint,
        }
        for key, value in expected.items():
            if key not in saved:
                raise RuntimeError(f"Cached split has no {key}: {split_path}")
            actual = saved[key].item()
            if isinstance(value, float):
                if not np.isclose(actual, value, rtol=0.0, atol=1e-12):
                    raise RuntimeError(f"Cached split {key} mismatch in {split_path}")
            elif actual != value:
                raise RuntimeError(f"Cached split {key} mismatch in {split_path}")
    else:
        rng = np.random.default_rng(seed)
        train_parts, validation_parts, test_parts = [], [], []
        for label in sorted(np.unique(labels).tolist()):
            indices = np.flatnonzero(labels == label)
            rng.shuffle(indices)
            n_test = max(1, int(round(len(indices) * test_fraction))) if test_fraction else 0
            n_val = max(1, int(round(len(indices) * val_fraction))) if val_fraction else 0
            if n_test + n_val >= len(indices):
                raise ValueError(
                    f"Class {label} has {len(indices)} samples, which cannot support "
                    f"train/validation/test counts {len(indices)-n_test-n_val}/{n_val}/{n_test}"
                )
            test_parts.append(indices[:n_test])
            validation_parts.append(indices[n_test:n_test + n_val])
            train_parts.append(indices[n_test + n_val:])
        train = np.concatenate(train_parts)
        validation = np.concatenate(validation_parts)
        test = np.concatenate(test_parts)
        rng.shuffle(train)
        rng.shuffle(validation)
        rng.shuffle(test)
        split_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = split_path.with_name(
            f"{split_path.stem}-building-{os.getpid()}.npz"
        )
        np.savez(
            temporary,
            train=train,
            val=validation,
            test=test,
            seed=seed,
            val_fraction=val_fraction,
            test_fraction=test_fraction,
            sample_fingerprint=sample_fingerprint,
        )
        os.replace(temporary, split_path)

    partitions = {
        "train": np.asarray(train, dtype=np.int64),
        "validation": np.asarray(validation, dtype=np.int64),
        "test": np.asarray(test, dtype=np.int64),
    }
    requested = {
        "train": True,
        "validation": val_fraction > 0.0,
        "test": test_fraction > 0.0,
    }
    for name, indices in partitions.items():
        if indices.ndim != 1:
            raise RuntimeError(f"{name} indices must be one-dimensional")
        if len(np.unique(indices)) != len(indices):
            raise RuntimeError(f"{name} contains duplicate indices")
        if requested[name] and len(indices) == 0:
            raise RuntimeError(f"Requested {name} partition is empty")
    names = tuple(partitions)
    for left_index, left_name in enumerate(names):
        for right_name in names[left_index + 1:]:
            if np.intersect1d(
                partitions[left_name], partitions[right_name], assume_unique=False
            ).size:
                raise RuntimeError(f"{left_name} and {right_name} overlap")
    combined = np.sort(np.concatenate(tuple(partitions.values())))
    if not np.array_equal(combined, np.arange(len(labels))):
        raise RuntimeError("Split does not cover every development sample exactly once")
    return partitions["train"], partitions["validation"], partitions["test"]


def _manifest_digests(cache_dir: str | Path) -> Dict[int, str]:
    with (Path(cache_dir) / "manifest.csv").open(newline="") as handle:
        rows = csv.DictReader(handle)
        return {int(row["index"]): row["sha256_visible"] for row in rows}


def require_content_disjoint_partitions(
    cache_dir: str | Path,
    partitions: Dict[str, np.ndarray],
) -> None:
    """Reject model-visible duplicate content assigned to different partitions."""

    digests = _manifest_digests(cache_dir)
    digest_sets = {
        name: {digests[int(index)] for index in indices}
        for name, indices in partitions.items()
    }
    names = tuple(digest_sets)
    for left_index, left_name in enumerate(names):
        for right_name in names[left_index + 1:]:
            overlap = digest_sets[left_name].intersection(digest_sets[right_name])
            if overlap:
                raise RuntimeError(
                    f"Model-visible content leak: {len(overlap)} digest(s) shared by "
                    f"{left_name} and {right_name}"
                )


def class_counts(labels: np.ndarray, indices: np.ndarray, names: List[str]) -> Dict[str, int]:
    selected = labels[np.asarray(indices, dtype=np.int64)]
    return {name: int((selected == label).sum()) for label, name in enumerate(names)}


def write_notebook_json_atomic(path: str | Path, value) -> None:
    """Write notebook orchestration metadata without depending on engine.py."""

    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(temporary, path)


def build_notebook_training_loaders(
    config: Config,
    seed: int,
    device: torch.device,
    test_root: str,
    test_fraction: float,
) -> Tuple[LoaderBundle, HeldoutTestPlan]:
    """Build leakage-safe training/validation loaders and an untouched test plan."""

    development_cache = config.cache_path / config.cache_key
    storage_dtype = np.float32 if config.dataset_id == "model_iv" else np.float16
    development_metadata = prepare_cache(
        config.development_path,
        development_cache,
        config.image_size,
        device,
        io_workers=config.io_workers,
        storage_dtype=storage_dtype,
    )
    labels = np.load(development_cache / "labels.npy")
    class_names = list(development_metadata["classes"])
    split_path = config.output_path / "split_indices.npz"
    sample_fingerprint = cache_identity_fingerprint(development_cache)

    if config.dataset_id in {"model_i", "model_ii", "model_iii"}:
        train_indices, validation_indices, unused_test = (
            fixed_stratified_train_validation_test_split(
                labels,
                split_path,
                val_fraction=config.val_fraction,
                test_fraction=0.0,
                seed=config.split_seed,
                sample_fingerprint=sample_fingerprint,
            )
        )
        assert len(unused_test) == 0
        validation_dataset = CachedNPYDataset(development_cache, validation_indices)
        heldout = HeldoutTestPlan(
            kind="official",
            class_names=class_names,
            cache_dir=config.cache_path / f"{config.cache_key}_official_test",
            source_root=test_root,
            description="separate official test root; unopened during training",
        )
        validation_mode = "fixed_80_20_development_split"
        disjoint = {"train": train_indices, "validation": validation_indices}
    elif config.dataset_id == "model_iv":
        train_indices, unused_validation, test_indices = (
            fixed_stratified_train_validation_test_split(
                labels,
                split_path,
                val_fraction=0.0,
                test_fraction=test_fraction,
                seed=config.split_seed,
                sample_fingerprint=sample_fingerprint,
            )
        )
        assert len(unused_validation) == 0
        validation_cache = config.cache_path / f"{config.cache_key}_validation"
        validation_metadata = prepare_cache(
            config.validation_path,
            validation_cache,
            config.image_size,
            device,
            io_workers=config.io_workers,
            storage_dtype=storage_dtype,
        )
        if class_names != list(validation_metadata["classes"]):
            raise RuntimeError("Model-IV development/validation class mappings differ")
        _require_disjoint_visible_content(development_cache, validation_cache)
        validation_labels = np.load(validation_cache / "labels.npy")
        validation_indices = np.arange(len(validation_labels), dtype=np.int64)
        validation_dataset = CachedNPYDataset(validation_cache, validation_indices)
        heldout = HeldoutTestPlan(
            kind="carved",
            class_names=class_names,
            cache_dir=development_cache,
            indices=test_indices,
            description="15% class-stratified holdout carved from Model-IV train/",
        )
        validation_mode = "supplied_development_validation"
        disjoint = {"train": train_indices, "test": test_indices}
    else:
        train_indices, validation_indices, test_indices = (
            fixed_stratified_train_validation_test_split(
                labels,
                split_path,
                val_fraction=config.val_fraction,
                test_fraction=test_fraction,
                seed=config.split_seed,
                sample_fingerprint=sample_fingerprint,
            )
        )
        validation_dataset = CachedNPYDataset(development_cache, validation_indices)
        heldout = HeldoutTestPlan(
            kind="carved",
            class_names=class_names,
            cache_dir=development_cache,
            indices=test_indices,
            description="15% test holdout from one Model-V 65/20/15 split",
        )
        validation_mode = "fixed_65_20_15_development_split"
        disjoint = {
            "train": train_indices,
            "validation": validation_indices,
            "test": test_indices,
        }

    require_content_disjoint_partitions(development_cache, disjoint)
    train_dataset = CachedNPYDataset(development_cache, train_indices)
    metadata = dict(development_metadata)
    metadata.update({
        "validation_mode": validation_mode,
        "test_mode": heldout.description,
        "split_seed": config.split_seed,
        "split_file": str(split_path),
        "sample_identity_sha256": sample_fingerprint,
        "training_sampler": {"kind": "random_batches"},
        "partition_counts": {
            "train": class_counts(labels, train_indices, class_names),
            "validation": (
                class_counts(labels, validation_indices, class_names)
                if config.dataset_id != "model_iv"
                else class_counts(
                    np.load(validation_dataset.cache_dir / "labels.npy"),
                    validation_indices,
                    class_names,
                )
            ),
            "test": (
                class_counts(labels, heldout.indices, class_names)
                if heldout.indices is not None
                else "separate_official_root_unopened"
            ),
        },
    })
    if config.dataset_id == "model_iv":
        metadata["validation"] = validation_metadata
        metadata["visible_content_overlap"] = 0
    training = LoaderBundle(
        train=make_loader(
            train_dataset,
            config.batch_size,
            shuffle=True,
            workers=config.workers,
            seed=seed,
        ),
        validation=make_loader(
            validation_dataset,
            config.batch_size,
            shuffle=False,
            workers=config.workers,
            seed=seed + 10_000,
        ),
        class_names=class_names,
        train_indices=train_indices,
        validation_indices=validation_indices,
        metadata=metadata,
    )
    return training, heldout


def create_model_iv_audit_training_view(
    config: Config,
    train_indices: np.ndarray,
) -> Path:
    """Create a fresh symlink view so the Model-IV audit cannot see carved test files."""

    samples, classes = list_samples(config.development_path)
    if classes != list(EXPECTED_CLASSES):
        raise RuntimeError("Unexpected Model-IV class mapping")
    view_root = config.output_path / "model_iv_audit_training_view"
    if view_root.exists():
        raise FileExistsError(f"Refusing to reuse audit view: {view_root}")
    for class_name in classes:
        (view_root / class_name).mkdir(parents=True, exist_ok=False)
    for index in np.asarray(train_indices, dtype=np.int64):
        source, _, relative = samples[int(index)]
        destination = view_root / relative
        destination.symlink_to(Path(source).resolve())
    return view_root


def build_final_test_loader(
    config: Config,
    plan: HeldoutTestPlan,
    device: torch.device,
) -> DataLoader:
    """Materialize the held-out loader only after validation-based selection."""

    if plan.kind == "official":
        if not plan.source_root.strip():
            raise ValueError("Set TEST_ROOT before the one-time official test evaluation")
        test_metadata = prepare_cache(
            Path(plan.source_root).expanduser().resolve(),
            plan.cache_dir,
            config.image_size,
            device,
            io_workers=config.io_workers,
            storage_dtype=np.float16,
        )
        if list(test_metadata["classes"]) != plan.class_names:
            raise RuntimeError("Official test and development class mappings differ")
        _require_disjoint_visible_content(
            config.cache_path / config.cache_key,
            plan.cache_dir,
        )
        labels = np.load(plan.cache_dir / "labels.npy")
        indices = np.arange(len(labels), dtype=np.int64)
    else:
        if plan.indices is None:
            raise RuntimeError("Carved test plan has no fixed indices")
        indices = plan.indices
    return make_loader(
        CachedNPYDataset(plan.cache_dir, indices),
        config.batch_size,
        shuffle=False,
        workers=config.workers,
        seed=config.split_seed + 20_000,
    )


## 4. Validate paths and establish the run contract

This creates the fresh top-level output directory. For Model IV it fixes
the test indices first and runs the source audit against a training-only
view before importing TorchQuantum or initializing CUDA. In explicit
`FINAL_TEST_ONLY` mode it instead validates and opens a completed run.


In [6]:
config = Config(
    dataset_id=DATASET_ID,
    development_root=DEVELOPMENT_ROOT,
    validation_root=VALIDATION_ROOT,
    cache_root=CACHE_ROOT,
    output_dir=OUTPUT_DIR,
    stage="all",
    allow_inconclusive_model_iv_audit=ALLOW_INCONCLUSIVE_MODEL_IV_AUDIT,
    quantum_epochs=QUANTUM_EPOCHS,
)
config.validate()

if config.dataset_id != "model_iv" and not np.isclose(
    config.val_fraction, 0.20, rtol=0.0, atol=1e-12
):
    raise ValueError("This notebook family fixes development validation at 20%")
if config.dataset_id in {"model_iv", "model_v"} and not np.isclose(
    TEST_FRACTION, 0.15, rtol=0.0, atol=1e-12
):
    raise ValueError("Models IV/V fix the carved test holdout at 15%")

if config.dataset_id in {"model_i", "model_ii", "model_iii"}:
    if VALIDATION_ROOT.strip():
        raise ValueError("Models I-III derive validation from DEVELOPMENT_ROOT")
elif config.dataset_id == "model_iv":
    if TEST_ROOT.strip():
        raise ValueError("Model IV has no supplied test root; leave TEST_ROOT empty")
else:
    if VALIDATION_ROOT.strip() or TEST_ROOT.strip():
        raise ValueError("Model V derives validation and test from DEVELOPMENT_ROOT")

if FINAL_TEST_ONLY:
    if not CONFIRM_FINAL_TEST_EVALUATION:
        raise ValueError(
            "FINAL_TEST_ONLY requires CONFIRM_FINAL_TEST_EVALUATION = True"
        )
    if not config.output_path.is_dir():
        raise FileNotFoundError(
            f"Completed OUTPUT_DIR does not exist: {config.output_path}"
        )
    contract_path = config.output_path / "notebook_run_contract.json"
    if not contract_path.is_file():
        raise FileNotFoundError(f"Completed-run contract is missing: {contract_path}")
    saved_contract = json.loads(contract_path.read_text())
    if saved_contract.get("dataset_id") != config.dataset_id:
        raise RuntimeError("Completed OUTPUT_DIR belongs to a different dataset")
    if int(saved_contract.get("split_seed", -1)) != config.split_seed:
        raise RuntimeError("Completed OUTPUT_DIR uses a different split seed")
    if int(saved_contract.get("quantum_epochs", -1)) != config.quantum_epochs:
        raise RuntimeError("Completed OUTPUT_DIR uses a different quantum epoch policy")
    print("Opened completed run for final-test-only evaluation:", config.output_path)
else:
    if config.output_path.exists():
        raise FileExistsError(
            f"Use a fresh OUTPUT_DIR; already exists: {config.output_path}"
        )
    config.output_path.mkdir(parents=True)

    # Model IV: fix the held-out test indices before the audit, then expose only the
    # remaining training files through a fresh symlink view. This keeps the source audit
    # CPU-only and prevents the carved test set from influencing the training gate.
    if config.dataset_id == "model_iv":
        audit_samples, _ = list_samples(config.development_path)
        audit_labels = np.asarray(
            [label for _, label, _ in audit_samples], dtype=np.int64
        )
        audit_train_indices, _, audit_test_indices = (
            fixed_stratified_train_validation_test_split(
                audit_labels,
                config.output_path / "split_indices.npz",
                val_fraction=0.0,
                test_fraction=TEST_FRACTION,
                seed=config.split_seed,
                sample_fingerprint=sample_identity_fingerprint(audit_samples),
            )
        )
        audit_view = create_model_iv_audit_training_view(
            config, audit_train_indices
        )
        audit_config = replace(config, development_root=str(audit_view))
        audit = run_model_iv_audit(audit_config, config.output_path)
        print(f"MODEL_IV_AUDIT_STATUS {audit.status} report={audit.report_path}")
        override = bool(
            audit.status == INCONCLUSIVE_NO_SIGNAL_DETECTED
            and config.allow_inconclusive_model_iv_audit
        )
        if audit.status != PASS_SIGNAL_DETECTED and not override:
            raise RuntimeError(
                f"Model-IV training gate did not pass; see {audit.report_path}"
            )
        if override:
            print(
                "MODEL_IV_AUDIT_OVERRIDE research_only=true "
                "status_remains_inconclusive=true"
            )

    write_notebook_json_atomic(
        config.output_path / "notebook_run_contract.json",
        {
            "dataset_id": config.dataset_id,
            "split_seed": config.split_seed,
            "validation_mode": (
                "supplied_root" if config.dataset_id == "model_iv" else "carved"
            ),
            "validation_fraction": (
                None if config.dataset_id == "model_iv" else config.val_fraction
            ),
            "test_mode": (
                "separate_official_root"
                if config.dataset_id in {"model_i", "model_ii", "model_iii"}
                else "carved_file_level_holdout"
            ),
            "test_fraction": (
                None
                if config.dataset_id in {"model_i", "model_ii", "model_iii"}
                else TEST_FRACTION
            ),
            "test_used_for_selection": False,
            "quantum_epochs": config.quantum_epochs,
        },
    )
    print("Runtime contract validated. Fresh output:", config.output_path)


MODEL_IV_AUDIT_SCAN development 5000/46497


MODEL_IV_AUDIT_SCAN development 10000/46497


MODEL_IV_AUDIT_SCAN development 15000/46497


MODEL_IV_AUDIT_SCAN development 20000/46497


MODEL_IV_AUDIT_SCAN development 25000/46497


...

MODEL_IV_AUDIT_SCAN validation 5000/6089


MODEL_IV_AUDIT_SCAN validation 6089/6089


MODEL_IV_AUDIT_PROBE_FEATURES view=raw dimension=141


MODEL_IV_AUDIT_PROBE_FEATURES view=model_visible dimension=141


MODEL_IV_AUDIT_STATUS INCONCLUSIVE_NO_SIGNAL_DETECTED report=<runtime-root>/outputs/model_iv/dataset_audit.md
MODEL_IV_AUDIT_OVERRIDE research_only=true status_remains_inconclusive=true
Runtime contract validated. Fresh output: <runtime-root>/outputs/model_iv


## 5. D4 lifting, morphology channels, and shared MBConv encoder

D4 equivariance is explicit: all eight rotations/reflections are lifted and
processed by one shared encoder. The deterministic mixed derivative is
feature engineering, not LensPINN and not a learned physics residual. The
source module's historical “Model-I encoder” label names the original run;
this same encoder is intentionally shared by Models I–V.


In [7]:
"""D4 orbit lifting and the selected shared Model-I image encoder."""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn


def norm2d(channels: int) -> nn.GroupNorm:
    """View-local normalization with identical train/evaluation behavior."""

    groups = min(8, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels)


def d4_transform(
    images: torch.Tensor, rotation: int, reflected: int
) -> torch.Tensor:
    """Apply ``r^rotation s^reflected`` to a batch of square images."""

    if reflected:
        images = torch.flip(images, dims=(-1,))
    return torch.rot90(images, rotation, dims=(-2, -1))


def d4_views(images: torch.Tensor) -> torch.Tensor:
    """Lift images to all eight D4 views in regular-representation order."""

    return torch.stack(
        [d4_transform(images, k, f) for f in (0, 1) for k in range(4)],
        dim=1,
    )


class MorphologyChannelBank(nn.Module):
    """Eight deterministic morphology channels for photon-count images.

    This is zero-parameter feature engineering, not a PINN.  No differential
    equation residual, lens inversion, or auxiliary loss is optimized.
    """

    output_channels = 8

    def __init__(
        self,
        log_gain: float = 20.0,
        epsilon: float = 1e-3,
        reference_pixels: int = 96,
    ) -> None:
        super().__init__()
        self.log_gain = float(log_gain)
        self.epsilon = float(epsilon)
        self.reference_pixels = int(reference_pixels)
        self.variant = "base"

        sobel_x = torch.tensor(
            [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]
        ) / 8.0
        sobel_y = sobel_x.transpose(0, 1).contiguous()
        laplacian = torch.tensor(
            [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]]
        )
        self.register_buffer(
            "sobel_x", sobel_x.view(1, 1, 3, 3), persistent=False
        )
        self.register_buffer(
            "sobel_y", sobel_y.view(1, 1, 3, 3), persistent=False
        )
        self.register_buffer(
            "laplacian", laplacian.view(1, 1, 3, 3), persistent=False
        )

    @staticmethod
    def _conv_reflect(images: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
        return F.conv2d(F.pad(images, (1, 1, 1, 1), mode="reflect"), kernel)

    @staticmethod
    def _avg_reflect(images: torch.Tensor, kernel_size: int) -> torch.Tensor:
        pad = kernel_size // 2
        return F.avg_pool2d(
            F.pad(images, (pad, pad, pad, pad), mode="reflect"),
            kernel_size=kernel_size,
            stride=1,
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        images = torch.nan_to_num(
            images.float(), nan=0.0, posinf=0.0, neginf=0.0
        ).clamp_min(0.0)
        scale = images.amax(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
        images = (images / scale).clamp(0.0, 1.0)
        log_intensity = torch.log1p(self.log_gain * images) / math.log1p(
            self.log_gain
        )

        gx = self._conv_reflect(log_intensity, self.sobel_x)
        gy = self._conv_reflect(log_intensity, self.sobel_y)
        pixel_scale = float(images.shape[-1]) / self.reference_pixels
        gradient = torch.sqrt(gx.square() + gy.square() + 1e-8) * pixel_scale
        laplacian = (
            self._conv_reflect(log_intensity, self.laplacian).abs()
            * pixel_scale**2
        )
        small_kernel = max(3, int(round(3 * pixel_scale)) | 1)
        large_kernel = max(small_kernel + 2, int(round(9 * pixel_scale)) | 1)
        dog = (
            self._avg_reflect(log_intensity, small_kernel)
            - self._avg_reflect(log_intensity, large_kernel)
        ).abs()

        height, width = images.shape[-2:]
        yy = torch.linspace(
            -1.0, 1.0, height, device=images.device, dtype=images.dtype
        )
        xx = torch.linspace(
            -1.0, 1.0, width, device=images.device, dtype=images.dtype
        )
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing="ij")
        radius = torch.sqrt(grid_x.square() + grid_y.square()).clamp_min(1e-4)
        unit_x = (grid_x / radius).view(1, 1, height, width)
        unit_y = (grid_y / radius).view(1, 1, height, width)
        radial = (gx * unit_x + gy * unit_y).abs() * pixel_scale
        tangential = (-gx * unit_y + gy * unit_x).abs() * pixel_scale

        # The final stabilized distortion channel is retained from the selected
        # run.  It is simply a fixed mixed finite derivative.
        log_ratio_sq = torch.log(
            (1.0 + self.epsilon) / (images + self.epsilon)
        ).square()
        mixed = self._conv_reflect(
            self._conv_reflect(log_ratio_sq, self.sobel_x), self.sobel_y
        ).abs() * pixel_scale**2

        return torch.cat(
            (
                images,
                log_intensity,
                torch.tanh(2.0 * gradient),
                torch.tanh(laplacian),
                torch.tanh(4.0 * dog),
                torch.tanh(2.0 * radial),
                torch.tanh(2.0 * tangential),
                torch.tanh(mixed),
            ),
            dim=1,
        )


class ForegroundSuppressedMorphologyChannelBank(MorphologyChannelBank):
    """Single-image Model-IV foreground suppression and SIS closure maps."""

    output_channels = 8

    def __init__(
        self,
        log_gain: float = 20.0,
        epsilon: float = 1e-3,
        reference_pixels: int = 96,
    ) -> None:
        super().__init__(
            log_gain=log_gain,
            epsilon=epsilon,
            reference_pixels=reference_pixels,
        )
        self.variant = "model_iv_sis_closure"
        axis = torch.tensor([-1.0, -2.0, 0.0, 2.0, 1.0])
        self.register_buffer(
            "mixed_derivative",
            torch.outer(axis, axis).view(1, 1, 5, 5) / 64.0,
            persistent=False,
        )

    @staticmethod
    def _geometry(
        height: int, width: int, device: torch.device
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if height != width:
            raise ValueError("D4 morphology requires square images")
        coordinates = torch.arange(height, device=device, dtype=torch.float32)
        coordinates = coordinates - (height - 1) / 2.0
        yy, xx = torch.meshgrid(coordinates, coordinates, indexing="ij")
        radius = torch.sqrt(xx.square() + yy.square())
        return radius, torch.floor(radius).long()

    @staticmethod
    def _border_location_scale(
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if min(images.shape[-2:]) < 16:
            raise ValueError(
                "Model-IV border estimation requires at least 16 pixels"
            )
        border = torch.cat(
            (
                images[..., :8, :].flatten(-2),
                images[..., -8:, :].flatten(-2),
                images[..., :, :8].flatten(-2),
                images[..., :, -8:].flatten(-2),
            ),
            dim=-1,
        )
        location = border.median(-1, keepdim=True).values
        mad = (border - location).abs().median(-1, keepdim=True).values
        return location[..., None], (1.4826 * mad).clamp_min(1e-7)[..., None]

    @staticmethod
    def _smooth_profile(
        profile: torch.Tensor, sigma: float = 0.8
    ) -> torch.Tensor:
        radius = max(1, int(4.0 * sigma + 0.5))
        offsets = torch.arange(
            -radius,
            radius + 1,
            device=profile.device,
            dtype=profile.dtype,
        )
        kernel = torch.exp(-0.5 * (offsets / sigma).square())
        kernel = (kernel / kernel.sum()).view(1, 1, -1)
        flat = profile.reshape(-1, 1, profile.shape[-1])
        return F.conv1d(
            F.pad(flat, (radius, radius), mode="replicate"), kernel
        ).reshape_as(profile)

    def _radial_median(self, features: torch.Tensor) -> torch.Tensor:
        batch, channels, height, width = features.shape
        _, radial_bin = self._geometry(height, width, features.device)
        flat = features.flatten(2)
        profile = torch.stack(
            [
                flat[..., radial_bin.flatten() == index].median(-1).values
                for index in range(int(radial_bin.max()) + 1)
            ],
            dim=-1,
        )
        profile = self._smooth_profile(profile)
        index = radial_bin.flatten().view(1, 1, -1).expand(
            batch, channels, -1
        )
        return profile.gather(2, index).reshape_as(features)

    @staticmethod
    def _normalize_map(
        features: torch.Tensor,
        radius: torch.Tensor,
        center_mask: bool = False,
    ) -> torch.Tensor:
        scale64 = features.shape[-1] / 64.0
        if center_mask:
            mask = (radius >= 3.0 * scale64) & (radius < 43.0 * scale64)
            values = features[..., mask].abs()
        else:
            values = features.flatten(2).abs()
        kth = max(1, int(math.ceil(0.99 * values.shape[-1])))
        scale = values.kthvalue(kth, dim=-1).values[..., None, None] + 1e-7
        output = (features / scale).clamp(-8.0, 8.0)
        if center_mask:
            output = output.masked_fill(
                (radius < 3.0 * scale64).view(1, 1, *radius.shape), 0.0
            )
        return output

    @staticmethod
    def _gaussian_filter(
        features: torch.Tensor, sigma: float
    ) -> torch.Tensor:
        radius = max(1, int(4.0 * sigma + 0.5))
        offsets = torch.arange(
            -radius,
            radius + 1,
            device=features.device,
            dtype=features.dtype,
        )
        kernel = torch.exp(-0.5 * (offsets / sigma).square())
        kernel = kernel / kernel.sum()
        channels = features.shape[1]
        kernel_x = kernel.view(1, 1, 1, -1).expand(
            channels, 1, 1, -1
        )
        kernel_y = kernel.view(1, 1, -1, 1).expand(
            channels, 1, -1, 1
        )
        output = F.conv2d(
            F.pad(features, (radius, radius, 0, 0), mode="reflect"),
            kernel_x,
            groups=channels,
        )
        return F.conv2d(
            F.pad(output, (0, 0, radius, radius), mode="reflect"),
            kernel_y,
            groups=channels,
        )

    @staticmethod
    def _estimate_einstein_radius(
        brightness: torch.Tensor, radius: torch.Tensor
    ) -> torch.Tensor:
        """Estimate one detached smooth-ring radius per supplied image.

        A soft radial peak is substantially cheaper than optimizing a lens
        model inside every training step.  The operation is label-free and
        depends on no other image in the batch.
        """

        scale = brightness.shape[-1] / 64.0
        candidates = torch.arange(
            7,
            28,
            device=brightness.device,
            dtype=brightness.dtype,
        ) * scale
        profiles = []
        positive = brightness.clamp_min(0.0)
        for candidate in candidates:
            mask = (radius >= candidate - 0.5 * scale) & (
                radius < candidate + 0.5 * scale
            )
            profiles.append(positive[..., mask].mean(-1))
        profile = torch.stack(profiles, dim=-1)
        profile = profile / profile.amax(-1, keepdim=True).clamp_min(1e-7)
        weights = torch.softmax(12.0 * profile, dim=-1)
        estimate = (weights * candidates.view(1, 1, -1)).sum(-1)
        return estimate[..., None, None].detach()

    @staticmethod
    def _sis_partner(
        features: torch.Tensor,
        einstein_radius: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Sample the conjugate SIS point from the same individual image."""

        _, _, height, width = features.shape
        if height != width:
            raise ValueError("SIS closure requires square images")
        axis = torch.arange(
            height, device=features.device, dtype=features.dtype
        ) - (height - 1) / 2.0
        yy, xx = torch.meshgrid(axis, axis, indexing="ij")
        radius = torch.sqrt(xx.square() + yy.square()).clamp_min(1e-4)
        factor = 1.0 - 2.0 * einstein_radius / radius.view(
            1, 1, height, width
        )
        partner_x = factor * xx.view(1, 1, height, width)
        partner_y = factor * yy.view(1, 1, height, width)
        grid = torch.stack(
            (
                2.0 * partner_x[:, 0] / max(width - 1, 1),
                2.0 * partner_y[:, 0] / max(height - 1, 1),
            ),
            dim=-1,
        )
        valid = (grid[..., 0].abs() <= 1.0) & (grid[..., 1].abs() <= 1.0)
        partner = F.grid_sample(
            features,
            grid,
            mode="bicubic",
            padding_mode="zeros",
            align_corners=True,
        )
        return partner, valid[:, None]

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        with torch.autocast(device_type=images.device.type, enabled=False):
            images = torch.nan_to_num(
                images.float(), nan=0.0, posinf=0.0, neginf=0.0
            )
            background, noise = self._border_location_scale(images)
            signal = images - background
            peak = signal.amax((-2, -1), keepdim=True).clamp_min(1e-7)
            current = (signal / peak).clamp(-1.0, 1.0)
            radius, _ = self._geometry(
                images.shape[-2], images.shape[-1], images.device
            )
            asinh_unscaled = torch.asinh(
                signal / (3.0 * noise).clamp_min(1e-6)
            )
            asinh_map = self._normalize_map(asinh_unscaled, radius)
            radial = self._normalize_map(
                signal - self._radial_median(signal), radius, True
            )
            asinh_radial = self._normalize_map(
                asinh_unscaled - self._radial_median(asinh_unscaled),
                radius,
                True,
            )
            size_scale = images.shape[-1] / 64.0
            smooth = tuple(
                self._gaussian_filter(asinh_unscaled, sigma * size_scale)
                for sigma in (0.8, 1.6, 3.2, 6.4)
            )
            dog_stack = torch.cat(
                tuple(
                    first - second
                    for first, second in zip(smooth[:-1], smooth[1:])
                ),
                dim=1,
            )
            einstein_radius = self._estimate_einstein_radius(
                asinh_map, radius
            )
            partner_stack, valid = self._sis_partner(
                torch.cat((dog_stack, asinh_unscaled), dim=1),
                einstein_radius,
            )
            partner_dogs = partner_stack[:, :3]
            partner_brightness = partner_stack[:, 3:4]
            union_brightness = (
                asinh_unscaled.clamp_min(0.0)
                + partner_brightness.clamp_min(0.0)
            )
            gate_scale = torch.quantile(
                union_brightness.flatten(2), 0.95, dim=-1
            )[..., None, None].clamp_min(1e-7)
            brightness_gate = (union_brightness / gate_scale).clamp(0.0, 1.0)
            offset = (
                radius.view(1, 1, *radius.shape) - einstein_radius
            ).abs()
            size_scale = images.shape[-1] / 64.0
            edge_width = max(0.5 * size_scale, 1e-3)
            closure_mask = (
                torch.sigmoid(
                    (offset - 2.0 * size_scale) / edge_width
                )
                * torch.sigmoid(
                    (8.0 * size_scale - offset) / edge_width
                )
                * valid
            )
            closure_dogs = torch.tanh(
                (dog_stack - partner_dogs) * brightness_gate
            ) * closure_mask
            unit = images / images.amax(
                (-2, -1), keepdim=True
            ).clamp_min(1e-7)
            unit = unit.clamp(0.0, 1.0)
            log_ratio_sq = torch.log(
                (1.0 + self.epsilon) / (unit + self.epsilon)
            ).square()
            mixed = F.conv2d(
                F.pad(log_ratio_sq, (2, 2, 2, 2), mode="reflect"),
                self.mixed_derivative,
            )
            mixed = mixed.abs()
            mixed = self._normalize_map(mixed, radius, True)
            return torch.cat(
                (
                    current,
                    asinh_map,
                    radial,
                    asinh_radial,
                    *closure_dogs.split(1, dim=1),
                    mixed,
                ),
                dim=1,
            )


class ModelIVPhysicsSummary(nn.Module):
    """1,395 zero-parameter D4-invariant Model-IV physics statistics."""

    output_dim = 1395
    RADIAL_EDGES = (0, 3, 5, 7, 9, 11, 13, 16, 20, 24, 29, 35, 46)
    MULTIPOLE_EDGES = (3, 6, 9, 12, 15, 19, 24, 31, 43)
    FOURIER_EDGES = (0, 1.5, 2.5, 3.5, 5, 7, 9, 12, 16, 21, 27, 34, 46)

    def __init__(self) -> None:
        super().__init__()
        sobel_x = torch.tensor(
            [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]
        ) / 8.0
        laplacian = torch.tensor(
            [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]]
        )
        mixed_axis = torch.tensor([-1.0, -2.0, 0.0, 2.0, 1.0])
        self.register_buffer(
            "summary_sobel_x",
            sobel_x.view(1, 1, 3, 3),
            persistent=False,
        )
        self.register_buffer(
            "summary_sobel_y",
            sobel_x.T.contiguous().view(1, 1, 3, 3),
            persistent=False,
        )
        self.register_buffer(
            "summary_laplacian",
            laplacian.view(1, 1, 3, 3),
            persistent=False,
        )
        self.register_buffer(
            "summary_mixed",
            torch.outer(mixed_axis, mixed_axis).view(1, 1, 5, 5) / 64.0,
            persistent=False,
        )

    @staticmethod
    def _geometry(
        height: int, width: int, device: torch.device
    ) -> tuple[torch.Tensor, torch.Tensor, float]:
        if height != width:
            raise ValueError("D4 summaries require square images")
        coordinates = torch.arange(height, device=device, dtype=torch.float32)
        coordinates = coordinates - (height - 1) / 2.0
        yy, xx = torch.meshgrid(coordinates, coordinates, indexing="ij")
        return (
            torch.sqrt(xx.square() + yy.square()),
            torch.atan2(yy, xx),
            height / 64.0,
        )

    @staticmethod
    def _masks(
        radius: torch.Tensor, edges, scale: float
    ) -> tuple[torch.Tensor, ...]:
        return tuple(
            (radius >= lower * scale) & (radius < upper * scale)
            for lower, upper in zip(edges[:-1], edges[1:])
        )

    @staticmethod
    def _scalar_annular(
        values: torch.Tensor, masks: tuple[torch.Tensor, ...]
    ) -> torch.Tensor:
        batch = values.shape[0]
        flat = values.reshape(batch, -1)
        quantiles = torch.tensor(
            (0.0, 0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75,
             0.90, 0.95, 0.99, 0.999, 1.0),
            device=values.device,
            dtype=values.dtype,
        )
        quantile_values = torch.quantile(flat, quantiles, dim=-1).T
        mean = flat.mean(-1)
        centered = flat - mean[:, None]
        std = (centered.square().mean(-1) + 1e-12).sqrt()
        moments = torch.stack(
            (
                mean,
                std,
                flat.abs().mean(-1),
                (flat.square().mean(-1) + 1e-12).sqrt(),
                centered.pow(3).mean(-1) / std.clamp_min(1e-6).pow(3),
                centered.pow(4).mean(-1) / std.clamp_min(1e-6).pow(4) - 3.0,
            ),
            -1,
        )
        output = [quantile_values, moments]
        annular_quantiles = torch.tensor(
            (0.10, 0.50, 0.90),
            device=values.device,
            dtype=values.dtype,
        )
        for mask in masks:
            annulus = values[..., mask]
            quantile = torch.quantile(
                annulus, annular_quantiles, dim=-1
            ).T
            absolute_99 = torch.quantile(
                annulus.abs(), 0.99, dim=-1, keepdim=True
            )
            output.append(
                torch.cat(
                    (
                        annulus.mean(-1, keepdim=True),
                        annulus.std(-1, unbiased=False, keepdim=True),
                        annulus.abs().mean(-1, keepdim=True),
                        (
                            annulus.square().mean(-1, keepdim=True) + 1e-12
                        ).sqrt(),
                        quantile,
                        absolute_99,
                    ),
                    -1,
                )
            )
        return torch.cat(output, -1)

    @staticmethod
    def _multipoles(
        values: torch.Tensor,
        theta: torch.Tensor,
        masks: tuple[torch.Tensor, ...],
    ) -> torch.Tensor:
        output = []
        for mask in masks:
            annulus = values[..., mask]
            angle = theta[mask]
            norm = annulus.abs().sum(-1).clamp_min(1e-8)
            for order in range(1, 9):
                real = (annulus * torch.cos(order * angle)).sum(-1) / norm
                imaginary = (
                    annulus * torch.sin(order * angle)
                ).sum(-1) / norm
                output.append(
                    torch.sqrt(real.square() + imaginary.square() + 1e-16)
                )
        return torch.stack(output, -1)

    @classmethod
    def _spectrum(cls, values: torch.Tensor) -> torch.Tensor:
        _, height, width = values.shape
        window_y = torch.hann_window(
            height, periodic=False, device=values.device, dtype=values.dtype
        )
        window_x = torch.hann_window(
            width, periodic=False, device=values.device, dtype=values.dtype
        )
        centered = values - values.mean((-2, -1), keepdim=True)
        power = torch.fft.fft2(
            centered * torch.outer(window_y, window_x)
        ).abs().square()
        frequency_y = torch.fft.fftfreq(height, device=values.device) * height
        frequency_x = torch.fft.fftfreq(width, device=values.device) * width
        yy, xx = torch.meshgrid(frequency_y, frequency_x, indexing="ij")
        radius = torch.sqrt(xx.square() + yy.square())
        theta = torch.atan2(yy, xx)
        scale = height / 64.0
        total = power.sum((-2, -1)).clamp_min(1e-10)
        output = [
            power[..., mask].sum(-1) / total
            for mask in cls._masks(radius, cls.FOURIER_EDGES, scale)
        ]
        for lower, upper in ((2, 6), (6, 12), (12, 22), (22, 40)):
            mask = (radius >= lower * scale) & (radius < upper * scale)
            annulus = power[..., mask]
            angle = theta[mask]
            norm = annulus.sum(-1).clamp_min(1e-10)
            for order in (2, 4, 6, 8):
                real = (annulus * torch.cos(order * angle)).sum(-1) / norm
                imaginary = (
                    annulus * torch.sin(order * angle)
                ).sum(-1) / norm
                output.append(
                    torch.sqrt(real.square() + imaginary.square() + 1e-20)
                )
        return torch.stack(output, -1)

    @staticmethod
    def _conv(values: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
        pad = kernel.shape[-1] // 2
        return F.conv2d(
            F.pad(values[:, None], (pad, pad, pad, pad), mode="reflect"),
            kernel,
        )[:, 0]

    def _derivatives(
        self,
        values: torch.Tensor,
        theta: torch.Tensor,
        masks: tuple[torch.Tensor, ...],
    ) -> torch.Tensor:
        gradient_x = self._conv(values, self.summary_sobel_x)
        gradient_y = self._conv(values, self.summary_sobel_y)
        gradient = torch.sqrt(
            gradient_x.square() + gradient_y.square() + 1e-12
        )
        radial = gradient_x * torch.cos(theta) + gradient_y * torch.sin(theta)
        tangential = (
            -gradient_x * torch.sin(theta) + gradient_y * torch.cos(theta)
        )
        laplacian = self._conv(values, self.summary_laplacian)
        mixed = self._conv(values, self.summary_mixed)
        output = []
        for mask in masks:
            for derivative in (
                gradient,
                radial,
                tangential,
                laplacian,
                mixed,
            ):
                annulus = derivative[..., mask]
                output.extend(
                    (
                        annulus.abs().mean(-1),
                        (annulus.square().mean(-1) + 1e-12).sqrt(),
                    )
                )
        return torch.stack(output, -1)

    def forward(self, morphology: torch.Tensor) -> torch.Tensor:
        if morphology.ndim != 4 or morphology.shape[1] != 8:
            raise ValueError(
                "Model-IV physics summaries require [B, 8, H, W] maps"
            )
        with torch.autocast(device_type=morphology.device.type, enabled=False):
            morphology = morphology.float()
            radius, theta, scale = self._geometry(
                morphology.shape[-2], morphology.shape[-1], morphology.device
            )
            annuli = self._masks(radius, self.RADIAL_EDGES, scale)
            multipole_annuli = self._masks(
                radius, self.MULTIPOLE_EDGES, scale
            )
            output = []
            for channel in range(2, 7):
                values = morphology[:, channel]
                output.extend(
                    (
                        self._scalar_annular(values, annuli),
                        self._multipoles(values, theta, multipole_annuli),
                        self._spectrum(values),
                    )
                )
                if channel >= 4:
                    output.append(self._derivatives(values, theta, annuli))
            summary = torch.cat(output, -1)
            if summary.shape[-1] != self.output_dim:
                raise RuntimeError(
                    f"Model-IV summary drift: {summary.shape[-1]} != "
                    f"{self.output_dim}"
                )
            return torch.nan_to_num(
                summary, nan=0.0, posinf=1e6, neginf=-1e6
            )


# Kept as a small source-compatibility alias for old imports.  The clearer
# class name above describes what the module actually does.
PhysicsChannelBank = MorphologyChannelBank


class SqueezeExcite(nn.Module):
    def __init__(self, channels: int, reduction: int = 4) -> None:
        super().__init__()
        hidden = max(8, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Hardsigmoid(inplace=True),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return features * self.net(features)


class MBConv(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        expand: int,
        stride: int,
        kernel_size: int = 5,
    ) -> None:
        super().__init__()
        hidden = in_channels * expand
        layers: list[nn.Module] = []
        if hidden != in_channels:
            layers.extend(
                (
                    nn.Conv2d(in_channels, hidden, 1, bias=False),
                    norm2d(hidden),
                    nn.SiLU(inplace=True),
                )
            )
        layers.extend(
            (
                nn.Conv2d(
                    hidden,
                    hidden,
                    kernel_size,
                    stride=stride,
                    padding=kernel_size // 2,
                    groups=hidden,
                    bias=False,
                ),
                norm2d(hidden),
                nn.SiLU(inplace=True),
                SqueezeExcite(hidden),
                nn.Conv2d(hidden, out_channels, 1, bias=False),
                norm2d(out_channels),
            )
        )
        self.block = nn.Sequential(*layers)
        self.use_residual = stride == 1 and in_channels == out_channels

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        output = self.block(features)
        return features + output if self.use_residual else output


class CompactOrbitEncoder(nn.Module):
    """The selected tiny MBConv encoder, shared across all eight D4 views."""

    output_dim = 128
    variant = "tiny"

    def __init__(self, input_channels: int = 8) -> None:
        super().__init__()
        specs = (
            (16, 24, 2, 2),
            (24, 24, 2, 1),
            (24, 40, 3, 2),
            (40, 40, 3, 1),
            (40, 64, 3, 2),
            (64, 64, 3, 1),
            (64, 96, 3, 2),
            (96, 96, 2, 1),
        )
        self.stem = nn.Sequential(
            nn.Conv2d(
                input_channels, 16, 5, stride=2, padding=2, bias=False
            ),
            norm2d(16),
            nn.SiLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            *(MBConv(cin, cout, expand, stride) for cin, cout, expand, stride in specs)
        )
        self.final = nn.Sequential(
            nn.Conv2d(96, self.output_dim, 1, bias=False),
            norm2d(self.output_dim),
            nn.SiLU(inplace=True),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.final(self.blocks(self.stem(images)))
        return features.mean(dim=(-2, -1))


## 6. D4 group algebra and TorchQuantum orbit circuit

Eight qubits are indexed by D4 elements. Gate parameters are tied across
complete Cayley edge orbits; one- and two-qubit orbit reductions produce 48
invariants. Autocast is disabled around complex statevector execution so
input and circuit-parameter gradients remain valid.


In [8]:
"""TorchQuantum implementation of the selected D4 orbit circuit.

The eight qubits are indexed by D4 group elements.  Parameters are tied over
complete left-Cayley edge orbits, so the circuit is equivariant to the regular
group action.  Orbit-averaged one- and two-qubit observables provide invariant
features to the classifier.
"""

from __future__ import annotations

from typing import Any, Dict, List, Sequence, Tuple

import torch
from torch import nn

try:
    import torchquantum as tq
except ImportError as error:  # Keep --help and classical pretraining importable.
    tq = None
    _TORCHQUANTUM_IMPORT_ERROR: ImportError | None = error
else:
    _TORCHQUANTUM_IMPORT_ERROR = None


D4Element = Tuple[int, int]
D4_ELEMENTS: Tuple[D4Element, ...] = tuple(
    (rotation, reflected) for reflected in (0, 1) for rotation in range(4)
)
D4_INDEX = {element: index for index, element in enumerate(D4_ELEMENTS)}


def d4_multiply(left: D4Element, right: D4Element) -> D4Element:
    """Multiply ``r^k s^f`` elements using ``s r s = r^-1``."""

    k, f = left
    ell, m = right
    return ((k + (-1 if f else 1) * ell) % 4, (f + m) % 2)


def right_regular_permutation(element: D4Element) -> torch.Tensor:
    """Return ``p`` such that an orbit field maps as ``z'(g)=z(g*h)``."""

    return torch.tensor(
        [D4_INDEX[d4_multiply(group, element)] for group in D4_ELEMENTS],
        dtype=torch.long,
    )


def _unique_undirected_edges(
    generator: D4Element,
) -> Tuple[Tuple[int, int], ...]:
    edges = set()
    for group in D4_ELEMENTS:
        left = D4_INDEX[group]
        right = D4_INDEX[d4_multiply(generator, group)]
        if left != right:
            edges.add(tuple(sorted((left, right))))
    return tuple(sorted(edges))


R_EDGES = _unique_undirected_edges((1, 0))
R2_EDGES = _unique_undirected_edges((2, 0))
S_EDGES = _unique_undirected_edges((0, 1))


def _bit_mask(qubit: int, n_qubits: int) -> int:
    """Return the big-endian statevector bit used by TorchQuantum wire IDs."""

    return 1 << (n_qubits - qubit - 1)


def require_torchquantum() -> Any:
    """Return TorchQuantum or fail before a requested quantum run starts."""

    if tq is None:
        raise ModuleNotFoundError(
            "TorchQuantum is required for the D4 quantum stage. Install the "
            "dependencies from src/requirements.txt before training."
        ) from _TORCHQUANTUM_IMPORT_ERROR
    return tq


def _rzz(qdev: Any, theta: torch.Tensor, first: int, second: int) -> None:
    """Apply ``exp(-i theta Z⊗Z / 2)``, using a portable fallback."""

    if hasattr(qdev, "rzz"):
        qdev.rzz(wires=[first, second], params=theta)
        return
    qdev.cnot(wires=[first, second])
    qdev.rz(wires=second, params=theta)
    qdev.cnot(wires=[first, second])


def _rxx(qdev: Any, theta: torch.Tensor, first: int, second: int) -> None:
    """Apply ``exp(-i theta X⊗X / 2)``, using a portable fallback."""

    if hasattr(qdev, "rxx"):
        qdev.rxx(wires=[first, second], params=theta)
        return
    qdev.h(wires=first)
    qdev.h(wires=second)
    _rzz(qdev, theta, first, second)
    qdev.h(wires=first)
    qdev.h(wires=second)


def _edge_expectation_z(
    probabilities: torch.Tensor,
    z_signs: torch.Tensor,
    edges: Sequence[Tuple[int, int]],
) -> torch.Tensor:
    observables = torch.stack([z_signs[a] * z_signs[b] for a, b in edges])
    return probabilities @ observables.transpose(0, 1)


def _expectation_x(
    state: torch.Tensor,
    qubits: Sequence[Tuple[int, ...]],
    n_qubits: int,
) -> torch.Tensor:
    values: List[torch.Tensor] = []
    basis = torch.arange(state.shape[1], device=state.device)
    for wires in qubits:
        mask = 0
        for wire in wires:
            mask |= _bit_mask(wire, n_qubits)
        flipped = state.index_select(1, basis ^ mask)
        values.append((state.conj() * flipped).sum(dim=1).real)
    return torch.stack(values, dim=1)


_QuantumModule = tq.QuantumModule if tq is not None else nn.Module


class D4OrbitQuantumBottleneck(_QuantumModule):
    """Batched eight-qubit D4-equivariant TorchQuantum circuit heads."""

    parameters_per_layer = 11
    invariants_per_head = 12

    def __init__(
        self, heads: int = 4, reuploads: int = 2, n_qubits: int = 8
    ) -> None:
        require_torchquantum()
        super().__init__()
        if n_qubits != len(D4_ELEMENTS):
            raise ValueError("The D4 regular register requires exactly 8 qubits")
        if heads < 1 or reuploads < 1:
            raise ValueError("heads and reuploads must both be positive")

        self.heads = heads
        self.reuploads = reuploads
        self.n_qubits = n_qubits
        self.input_encoding = "angle"
        self.observable_readout = "pair"

        parameters = torch.zeros(heads, reuploads, self.parameters_per_layer)
        parameters[..., 0] = 1.0
        parameters[..., 2] = 1.0
        parameters[..., 4:] = 0.02 * torch.randn_like(parameters[..., 4:])
        self.params = nn.Parameter(parameters)

        basis = torch.arange(1 << n_qubits)
        signs = []
        for qubit in range(n_qubits):
            bit = (basis & _bit_mask(qubit, n_qubits)) != 0
            signs.append(
                torch.where(bit, -torch.ones_like(basis), torch.ones_like(basis))
            )
        self.register_buffer(
            "z_signs", torch.stack(signs).float(), persistent=False
        )

        # QuantumDevice owns transient execution state, not learned model state.
        self._qdev_cache: Dict[Tuple[int, str], Any] = {}

    @property
    def output_dim(self) -> int:
        return self.heads * self.invariants_per_head

    def _quantum_device(self, batch: int, device: torch.device) -> Any:
        key = (batch, str(device))
        qdev = self._qdev_cache.get(key)
        if qdev is None:
            torchquantum = require_torchquantum()
            qdev = torchquantum.QuantumDevice(
                n_wires=self.n_qubits,
                bsz=batch,
                device=device,
                record_op=False,
            )
            self._qdev_cache[key] = qdev
        else:
            qdev.reset_states(batch)
        return qdev

    @staticmethod
    def _edge_rotations(
        qdev: Any,
        theta: torch.Tensor,
        edges: Sequence[Tuple[int, int]],
        pauli: str,
    ) -> None:
        if pauli == "z":
            operation = _rzz
        elif pauli == "x":
            operation = _rxx
        else:
            raise ValueError(f"Unsupported Pauli rotation: {pauli}")
        for first, second in edges:
            operation(qdev, theta, first, second)

    def _run_statevector(self, orbit_features: torch.Tensor) -> torch.Tensor:
        batch = orbit_features.shape[0]
        flat = orbit_features.reshape(batch * self.heads, 2, self.n_qubits).float()
        parameters = self.params.unsqueeze(0).expand(batch, -1, -1, -1)
        parameters = parameters.reshape(
            batch * self.heads,
            self.reuploads,
            self.parameters_per_layer,
        ).float()

        qdev = self._quantum_device(flat.shape[0], flat.device)
        for layer in range(self.reuploads):
            layer_parameters = parameters[:, layer]

            # Data re-upload: two orbit channels become RY and RZ angles.
            for qubit in range(self.n_qubits):
                ry_angle = (
                    layer_parameters[:, 0] * flat[:, 0, qubit]
                    + layer_parameters[:, 1]
                )
                rz_angle = (
                    layer_parameters[:, 2] * flat[:, 1, qubit]
                    + layer_parameters[:, 3]
                )
                qdev.ry(wires=qubit, params=ry_angle)
                qdev.rz(wires=qubit, params=rz_angle)

            # Shared single-qubit trainable gates.
            for qubit in range(self.n_qubits):
                qdev.rx(wires=qubit, params=layer_parameters[:, 4])
                qdev.ry(wires=qubit, params=layer_parameters[:, 5])
                qdev.rz(wires=qubit, params=layer_parameters[:, 6])

            # Complete rotation/reflection Cayley edge orbits.
            self._edge_rotations(
                qdev, layer_parameters[:, 7], R_EDGES, "z"
            )
            self._edge_rotations(
                qdev, layer_parameters[:, 8], S_EDGES, "z"
            )
            self._edge_rotations(
                qdev, layer_parameters[:, 9], R_EDGES, "x"
            )
            self._edge_rotations(
                qdev, layer_parameters[:, 10], S_EDGES, "x"
            )

        return qdev.get_states_1d()

    def forward(
        self, orbit_features: torch.Tensor, return_equivariant: bool = False
    ) -> torch.Tensor | Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        expected = (self.heads, 2, self.n_qubits)
        if orbit_features.ndim != 4 or orbit_features.shape[1:] != expected:
            raise ValueError(
                f"Expected (B,{self.heads},2,{self.n_qubits}), "
                f"got {tuple(orbit_features.shape)}"
            )

        with torch.autocast(
            device_type=orbit_features.device.type, enabled=False
        ):
            state = self._run_statevector(orbit_features)
            probabilities = state.abs().square()
            z = probabilities @ self.z_signs.transpose(0, 1)
            x = _expectation_x(
                state,
                [(qubit,) for qubit in range(self.n_qubits)],
                self.n_qubits,
            )
            edge_families = (R_EDGES, R2_EDGES, S_EDGES)
            zz = tuple(
                _edge_expectation_z(probabilities, self.z_signs, edges)
                for edges in edge_families
            )
            xx = tuple(
                _expectation_x(state, edges, self.n_qubits)
                for edges in edge_families
            )

            def edge_product(
                values: torch.Tensor, edges: Sequence[Tuple[int, int]]
            ) -> torch.Tensor:
                return torch.stack(
                    [values[:, a] * values[:, b] for a, b in edges], dim=1
                )

            z_mean = z.mean(dim=1)
            x_mean = x.mean(dim=1)
            invariant_features = [
                z_mean,
                z.square().mean(dim=1) - z_mean.square(),
                x_mean,
                x.square().mean(dim=1) - x_mean.square(),
                *(values.mean(dim=1) for values in zz),
                *(values.mean(dim=1) for values in xx),
                (zz[0] - edge_product(z, R_EDGES)).mean(dim=1),
                (xx[0] - edge_product(x, R_EDGES)).mean(dim=1),
            ]
            invariant = torch.stack(invariant_features, dim=1).reshape(
                orbit_features.shape[0], self.output_dim
            )

        if not return_equivariant:
            return invariant
        equivariant = {
            "z": z.reshape(
                orbit_features.shape[0], self.heads, self.n_qubits
            ),
            "x": x.reshape(
                orbit_features.shape[0], self.heads, self.n_qubits
            ),
        }
        return invariant, equivariant

    def parameter_report(self) -> Dict[str, int | str]:
        return {
            "qubits": self.n_qubits,
            "heads": self.heads,
            "reuploads": self.reuploads,
            "quantum_trainable": self.params.numel(),
            "input_encoding": self.input_encoding,
            "observable_readout": self.observable_readout,
            "execution_backend": "torchquantum",
            "statevector_dimension": 1 << self.n_qubits,
            "invariants": self.output_dim,
        }


def smoke_test_torchquantum(device: torch.device) -> Dict[str, int | str]:
    """Fail fast on backend, device, forward, and autograd incompatibility."""

    circuit = D4OrbitQuantumBottleneck().to(device)
    features = torch.linspace(
        -0.3, 0.3, 4 * 2 * 8, device=device, dtype=torch.float32
    ).reshape(1, 4, 2, 8)
    features.requires_grad_(True)
    output = circuit(features)
    output.square().mean().backward()
    gradients = (features.grad, circuit.params.grad)
    if not bool(torch.isfinite(output).all()) or any(
        gradient is None or not bool(torch.isfinite(gradient).all())
        for gradient in gradients
    ):
        raise RuntimeError("TorchQuantum forward/backward smoke test failed")
    return {
        "backend": "torchquantum",
        "device": str(device),
        "quantum_parameters": circuit.params.numel(),
        "invariant_features": circuit.output_dim,
    }


## 7. Classical pretraining core and complete hybrid classifier

The first stage uses a parameter-matched classical orbit mixer plus context
branch. The quantum stage starts with a fresh circuit and main classifier
while loading the selected source-defined backbone state. Model IV alone
activates the foreground-suppressed morphology and invariant summary head;
the existing source checkpoint contract transfers that auxiliary head.


In [9]:
"""Composition of the selected shared encoder and D4 orbit bottlenecks."""

from __future__ import annotations

import math
from typing import Dict, Literal, Sequence, Tuple

import torch
from torch import nn



def _edge_products(
    values: torch.Tensor, edges: Sequence[Tuple[int, int]]
) -> torch.Tensor:
    return torch.stack(
        [values[..., first] * values[..., second] for first, second in edges],
        dim=-1,
    )


class ClassicalOrbitMixer(nn.Module):
    """Parameter-matched classical scaffold used only for backbone pretraining."""

    invariants_per_head = 12

    def __init__(self, heads: int = 4, layers: int = 2) -> None:
        super().__init__()
        self.heads = heads
        self.layers = layers
        parameters = torch.zeros(heads, layers, 11)
        parameters[..., 0] = 1.0
        parameters[..., 6] = 1.0
        parameters[..., 10] = 1.0
        parameters += 0.02 * torch.randn_like(parameters)
        self.params = nn.Parameter(parameters)

    @property
    def output_dim(self) -> int:
        return self.heads * self.invariants_per_head

    def forward(self, orbit_features: torch.Tensor) -> torch.Tensor:
        first, second = orbit_features[:, :, 0], orbit_features[:, :, 1]
        for layer in range(self.layers):
            parameters = self.params[:, layer].unsqueeze(0)
            new_first = torch.tanh(
                parameters[..., 0, None] * first
                + parameters[..., 1, None] * second
                + parameters[..., 2, None]
                + parameters[..., 3, None] * torch.sin(first)
                + parameters[..., 4, None] * torch.cos(second)
            )
            new_second = torch.tanh(
                parameters[..., 5, None] * first
                + parameters[..., 6, None] * second
                + parameters[..., 7, None]
                + parameters[..., 8, None] * torch.sin(second)
                + parameters[..., 9, None] * torch.cos(first)
            )
            residual = torch.sigmoid(parameters[..., 10, None])
            first = residual * first + (1.0 - residual) * new_first
            second = residual * second + (1.0 - residual) * new_second

        first_mean = first.mean(-1)
        second_mean = second.mean(-1)
        invariant_features = [
            first_mean,
            first.square().mean(-1) - first_mean.square(),
            second_mean,
            second.square().mean(-1) - second_mean.square(),
            _edge_products(first, R_EDGES).mean(-1),
            _edge_products(first, R2_EDGES).mean(-1),
            _edge_products(first, S_EDGES).mean(-1),
            _edge_products(second, R_EDGES).mean(-1),
            _edge_products(second, R2_EDGES).mean(-1),
            _edge_products(second, S_EDGES).mean(-1),
            (
                _edge_products(first, R_EDGES)
                - first_mean[..., None].square()
            ).mean(-1),
            (
                _edge_products(second, R_EDGES)
                - second_mean[..., None].square()
            ).mean(-1),
        ]
        features = torch.stack(invariant_features, dim=-1)
        return features.reshape(orbit_features.shape[0], self.output_dim)


class D4OrbitClassifier(nn.Module):
    """Selected eight-view D4-ORQB Model-I classifier."""

    def __init__(
        self,
        num_classes: int = 3,
        heads: int = 4,
        reuploads: int = 2,
        core: Literal["quantum", "classical"] = "quantum",
        include_context: bool = False,
        dropout: float = 0.10,
        foreground_suppressed: bool = False,
    ) -> None:
        super().__init__()
        if num_classes != 3:
            raise ValueError("The selected Model-I classifier has three classes")
        self.heads = heads
        self.include_context = include_context
        self.core_name = core

        # These remain top-level modules so the historical backbone-prefix
        # initialization contract stays stable.
        self.physics = (
            ForegroundSuppressedMorphologyChannelBank()
            if foreground_suppressed
            else MorphologyChannelBank()
        )
        if foreground_suppressed:
            self.physics_summary: nn.Module | None = ModelIVPhysicsSummary()
            self.physics_summary_norm: nn.Module | None = nn.LayerNorm(
                ModelIVPhysicsSummary.output_dim,
                elementwise_affine=False,
            )
            self.physics_summary_head: nn.Module | None = nn.Linear(
                ModelIVPhysicsSummary.output_dim, num_classes
            )
            nn.init.zeros_(self.physics_summary_head.weight)
            nn.init.zeros_(self.physics_summary_head.bias)
        else:
            self.physics_summary = None
            self.physics_summary_norm = None
            self.physics_summary_head = None
        self.encoder = CompactOrbitEncoder(
            input_channels=self.physics.output_channels
        )
        self.orbit_projection = nn.Linear(self.encoder.output_dim, heads * 2)

        if core == "quantum":
            self.core: nn.Module = D4OrbitQuantumBottleneck(
                heads=heads, reuploads=reuploads
            )
        elif core == "classical":
            self.core = ClassicalOrbitMixer(heads=heads, layers=reuploads)
        else:
            raise ValueError(f"Unknown core: {core}")

        context_dim = self.encoder.output_dim * 3 if include_context else 0
        if include_context:
            self.context_projection: nn.Module | None = nn.Sequential(
                nn.LayerNorm(context_dim),
                nn.Linear(context_dim, 64),
                nn.SiLU(inplace=True),
            )
            context_dim = 64
        else:
            self.context_projection = None

        head_input = self.core.output_dim + context_dim
        self.head = nn.Sequential(
            nn.LayerNorm(head_input),
            nn.Linear(head_input, 32),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes),
        )

    def orbit_encode(
        self,
        images: torch.Tensor,
        morphology: torch.Tensor | None = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return learned orbit embeddings and circuit angle features."""

        if self.physics.variant.startswith("model_iv_"):
            # The deterministic bank is D4-equivariant, so lifting its output
            # is exactly equivalent and avoids evaluating it eight times.
            if morphology is None:
                morphology = self.physics(images)
            morphology_views = d4_views(morphology)
            batch, group, channels, height, width = morphology_views.shape
            morphology = morphology_views.reshape(
                batch * group, channels, height, width
            )
        else:
            views = d4_views(images)
            batch, group, channels, height, width = views.shape
            flat_views = views.reshape(batch * group, channels, height, width)
            morphology = self.physics(flat_views)
        flat_encoded = self.encoder(morphology)
        encoded = flat_encoded.reshape(batch, group, -1)
        projected = self.orbit_projection(flat_encoded)
        projected = projected.reshape(batch, group, self.heads, 2).permute(
            0, 2, 3, 1
        )
        angles = math.pi * torch.tanh(projected)
        return encoded, angles

    def forward(self, images: torch.Tensor, return_aux: bool = False):
        # Canonicalize D4-transformed views before deterministic physics and
        # convolution kernels. Odd torch.rot90 actions otherwise retain a
        # transposed stride layout that can select numerically different CUDA
        # kernels and amplify roundoff after feature standardization. Plain
        # contiguous format is unambiguous for the singleton input channel.
        images = images.contiguous()
        morphology = None
        summary = None
        if self.physics_summary is not None:
            morphology = self.physics(images)
            summary = self.physics_summary(morphology)
        encoded, angles = self.orbit_encode(images, morphology=morphology)
        context_embedding = None
        if self.context_projection is not None:
            context = torch.cat(
                (
                    encoded.mean(dim=1),
                    encoded.std(dim=1, unbiased=False),
                    encoded.amax(dim=1),
                ),
                dim=1,
            )
            context_embedding = self.context_projection(context)

        if return_aux and self.core_name == "quantum":
            invariants, equivariant = self.core(
                angles, return_equivariant=True
            )
        else:
            invariants = self.core(angles)
            equivariant = None
        features = [invariants]
        if context_embedding is not None:
            features.append(context_embedding)
        logits = self.head(torch.cat(features, dim=1))
        summary_logits = None
        if summary is not None:
            assert self.physics_summary_norm is not None
            assert self.physics_summary_head is not None
            summary_logits = self.physics_summary_head(
                self.physics_summary_norm(summary)
            )
            logits = logits + summary_logits

        if return_aux:
            return logits, {
                "encoded": encoded,
                "angles": angles,
                "invariants": invariants,
                "equivariant": equivariant,
                "physics_summary": summary,
                "physics_summary_logits": summary_logits,
            }
        return logits

    def parameter_report(self) -> Dict[str, int | str]:
        def count(module: nn.Module) -> int:
            return sum(
                parameter.numel()
                for parameter in module.parameters()
                if parameter.requires_grad
            )

        context = (
            count(self.context_projection)
            if self.context_projection is not None
            else 0
        )
        summary_parameters = (
            count(self.physics_summary_head)
            if self.physics_summary_head is not None
            else 0
        )
        return {
            "total": count(self),
            "morphology_channels": count(self.physics),
            "morphology_variant": self.physics.variant,
            "physics_summary_dim": (
                ModelIVPhysicsSummary.output_dim
                if self.physics_summary is not None
                else 0
            ),
            "physics_summary_head": summary_parameters,
            "encoder": count(self.encoder),
            "orbit_projection": count(self.orbit_projection),
            "core": count(self.core),
            "head_and_context": count(self.head) + context,
            "core_architecture": self.core_name,
            "encoder_variant": "tiny",
            "encoder_output_dim": self.encoder.output_dim,
            "input_channels": self.physics.output_channels,
            "quantum_encoding": "angle",
            "observable_readout": "pair",
            "execution_backend": (
                "torchquantum" if self.core_name == "quantum" else "classical"
            ),
        }


def build_model(
    config: Config,
    core: Literal["quantum", "classical"] = "quantum",
    include_context: bool = False,
) -> D4OrbitClassifier:
    """Build one of the two fixed stages without embedding device policy."""

    return D4OrbitClassifier(
        num_classes=3,
        heads=config.heads,
        reuploads=config.reuploads,
        core=core,
        include_context=include_context,
        dropout=config.dropout,
        foreground_suppressed=config.dataset_id == "model_iv",
    )


def parameter_summary(model: D4OrbitClassifier) -> Dict[str, int | str]:
    return model.parameter_report()


## 8. Metrics, checkpoint transfer, symmetry audit, and training engine

Selection is lexicographic on development validation: balanced accuracy,
macro one-vs-rest AUC, then negative NLL. The engine saves `last.pt`, updates
`best.pt` only on validation improvement, reloads the best model, and records
a full eight-action D4 audit. Test data is not an engine input.


In [10]:
"""Training, validation, checkpointing, metrics, and symmetry audits."""

from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Literal, Tuple

import numpy as np
import torch
import torch.nn.functional as F



BACKBONE_PREFIXES = (
    "physics.",
    "physics_summary.",
    "physics_summary_norm.",
    "physics_summary_head.",
    "encoder.",
    "orbit_projection.",
)


@dataclass(frozen=True, slots=True)
class StageSpec:
    name: str
    core: Literal["quantum", "classical"]
    include_context: bool
    epochs: int
    patience: int
    seed: int
    encoder_learning_rate: float
    learning_rate: float
    core_learning_rate: float


def pretrain_spec(config: Config) -> StageSpec:
    return StageSpec(
        name="pretrain_context",
        core="classical",
        include_context=True,
        epochs=config.pretrain_epochs,
        patience=config.pretrain_patience,
        seed=config.pretrain_seed,
        encoder_learning_rate=config.pretrain_learning_rate,
        learning_rate=config.pretrain_learning_rate,
        core_learning_rate=config.pretrain_core_learning_rate,
    )


def quantum_spec(config: Config) -> StageSpec:
    return StageSpec(
        name=f"quantum_seed{config.quantum_seed}_{config.quantum_epochs}ep",
        core="quantum",
        include_context=False,
        epochs=config.quantum_epochs,
        patience=config.quantum_patience,
        seed=config.quantum_seed,
        encoder_learning_rate=config.encoder_learning_rate,
        learning_rate=config.learning_rate,
        core_learning_rate=config.core_learning_rate,
    )


def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def confusion_matrix(
    labels: np.ndarray, predictions: np.ndarray, classes: int
) -> np.ndarray:
    matrix = np.zeros((classes, classes), dtype=np.int64)
    np.add.at(matrix, (labels, predictions), 1)
    return matrix


def binary_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(bool)
    positives = int(labels.sum())
    negatives = int((~labels).sum())
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    ordered_labels = labels[order]
    ordered_scores = scores[order]
    distinct = np.r_[
        np.flatnonzero(np.diff(ordered_scores)), len(ordered_scores) - 1
    ]
    true_positive = np.cumsum(ordered_labels)[distinct]
    false_positive = 1 + distinct - true_positive
    true_positive_rate = np.r_[0.0, true_positive / positives]
    false_positive_rate = np.r_[0.0, false_positive / negatives]
    trapezoid = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return float(trapezoid(true_positive_rate, false_positive_rate))


def _binary_roc_curve(
    labels: np.ndarray, scores: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """Return false/true-positive rates using the same ties as ``binary_auc``."""

    labels = np.asarray(labels).astype(bool)
    scores = np.asarray(scores)
    positives = int(labels.sum())
    negatives = int((~labels).sum())
    if labels.size == 0 or positives == 0 or negatives == 0:
        return np.empty(0, dtype=np.float64), np.empty(0, dtype=np.float64)
    order = np.argsort(-scores, kind="mergesort")
    ordered_labels = labels[order]
    ordered_scores = scores[order]
    distinct = np.r_[
        np.flatnonzero(np.diff(ordered_scores)), len(ordered_scores) - 1
    ]
    true_positive = np.cumsum(ordered_labels)[distinct]
    false_positive = 1 + distinct - true_positive
    true_positive_rate = np.r_[0.0, true_positive / positives]
    false_positive_rate = np.r_[0.0, false_positive / negatives]
    return false_positive_rate, true_positive_rate


def expected_calibration_error(
    probabilities: np.ndarray, labels: np.ndarray, bins: int = 15
) -> float:
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == labels
    edges = np.linspace(0.0, 1.0, bins + 1)
    calibration_error = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            calibration_error += mask.mean() * abs(
                float(correct[mask].mean()) - float(confidence[mask].mean())
            )
    return float(calibration_error)


def classification_metrics(
    labels: np.ndarray, logits: np.ndarray, class_names: List[str]
) -> Dict:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponent = np.exp(shifted)
    probabilities = exponent / exponent.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(axis=1)
    classes = len(class_names)
    matrix = confusion_matrix(labels, predictions, classes)
    per_class = {}
    f1_values, recalls, auc_values = [], [], []
    for label, name in enumerate(class_names):
        true_positive = int(matrix[label, label])
        false_positive = int(matrix[:, label].sum() - true_positive)
        false_negative = int(matrix[label, :].sum() - true_positive)
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        auc = binary_auc(labels == label, probabilities[:, label])
        per_class[name] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "auc_ovr": auc,
            "support": int(matrix[label].sum()),
        }
        f1_values.append(f1)
        recalls.append(recall)
        auc_values.append(auc)

    clipped = np.clip(
        probabilities[np.arange(len(labels)), labels], 1e-12, 1.0
    )
    one_hot = np.eye(classes, dtype=np.float64)[labels]
    result = {
        "samples": int(len(labels)),
        "accuracy": float((predictions == labels).mean()),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1_values)),
        "macro_auc_ovr": float(np.nanmean(auc_values)),
        "nll": float(-np.log(clipped).mean()),
        "brier": float(
            np.square(probabilities - one_hot).sum(axis=1).mean()
        ),
        "ece_15": expected_calibration_error(probabilities, labels, bins=15),
        "confusion_matrix": matrix.tolist(),
        "per_class": per_class,
    }
    return result


def _atomic_json(path: Path, value) -> None:
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(temporary, path)


def _atomic_checkpoint(path: Path, value) -> None:
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    torch.save(value, temporary)
    os.replace(temporary, path)


def _atomic_text(path: Path, value: str) -> None:
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    temporary.write_text(value)
    os.replace(temporary, path)


def _save_validation_roc_curve(
    path: Path,
    labels: np.ndarray,
    logits: np.ndarray,
    class_names: List[str],
    metrics: Dict,
) -> None:
    """Save development-validation one-vs-rest ROC curves without sklearn."""

    import matplotlib

    matplotlib.use("Agg", force=True)
    from matplotlib import pyplot as plt

    shifted = logits - logits.max(axis=1, keepdims=True)
    exponent = np.exp(shifted)
    probabilities = exponent / exponent.sum(axis=1, keepdims=True)

    figure, axis = plt.subplots(figsize=(6.4, 5.2))
    try:
        for class_index, class_name in enumerate(class_names):
            false_positive_rate, true_positive_rate = _binary_roc_curve(
                labels == class_index, probabilities[:, class_index]
            )
            if false_positive_rate.size == 0:
                continue
            auc = metrics["per_class"][class_name]["auc_ovr"]
            axis.plot(
                false_positive_rate,
                true_positive_rate,
                linewidth=2,
                label=f"{class_name} (AUC = {auc:.4f})",
            )
        axis.plot(
            (0.0, 1.0),
            (0.0, 1.0),
            color="black",
            linestyle="--",
            linewidth=1,
            label="Chance",
        )
        axis.set(
            xlim=(0.0, 1.0),
            ylim=(0.0, 1.0),
            xlabel="False positive rate",
            ylabel="True positive rate",
            title="Development-validation one-vs-rest ROC",
        )
        axis.grid(alpha=0.25)
        axis.legend(loc="lower right")
        figure.tight_layout()
        temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
        figure.savefig(temporary, format="png", dpi=160)
        os.replace(temporary, path)
    finally:
        plt.close(figure)


def _write_validation_artifacts(
    output_dir: Path,
    stage: StageSpec,
    best_epoch: int,
    metrics: Dict,
    labels: np.ndarray,
    logits: np.ndarray,
    class_names: List[str],
) -> None:
    """Write concise, explicitly development-only final validation results."""

    _save_validation_roc_curve(
        output_dir / "validation_roc_curve.png",
        labels,
        logits,
        class_names,
        metrics,
    )
    lines = [
        "# Development-validation metrics",
        "",
        f"- Stage: `{stage.name}`",
        f"- Selected epoch: {best_epoch}",
        f"- Validation samples: {metrics['samples']}",
        f"- Accuracy: {metrics['accuracy']:.6f}",
        f"- Macro one-vs-rest AUC: {metrics['macro_auc_ovr']:.6f}",
        "- Official test evaluated: **No.** The official test set was not opened or evaluated.",
        "",
        "## Per-class one-vs-rest AUC",
        "",
        "| Class | AUC |",
        "| --- | ---: |",
    ]
    lines.extend(
        f"| {class_name} | "
        f"{metrics['per_class'][class_name]['auc_ovr']:.6f} |"
        for class_name in class_names
    )
    lines.extend(
        (
            "",
            "See `validation_roc_curve.png` for the corresponding ROC curves.",
            "",
        )
    )
    _atomic_text(output_dir / "validation_metrics.md", "\n".join(lines))


def load_backbone_checkpoint(
    model: D4OrbitClassifier, checkpoint_path: str | Path
) -> Dict:
    """Load only the fixed preprocessing, encoder, and orbit projection."""

    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    source = checkpoint.get("model", checkpoint.get("state_dict", checkpoint))
    if not isinstance(source, dict):
        raise RuntimeError(f"Invalid checkpoint state in {checkpoint_path}")

    target = model.state_dict()
    expected = {
        key
        for key in target
        if any(key.startswith(prefix) for prefix in BACKBONE_PREFIXES)
    }
    selected = {
        key: value
        for key, value in source.items()
        if key in expected and tuple(value.shape) == tuple(target[key].shape)
    }
    missing = sorted(expected.difference(selected))
    if missing:
        raise RuntimeError(
            "Backbone checkpoint is incomplete; missing compatible tensors: "
            f"{missing[:8]}"
        )
    target.update(selected)
    model.load_state_dict(target, strict=True)
    return {
        "checkpoint": str(checkpoint_path.resolve()),
        "loaded_prefixes": list(BACKBONE_PREFIXES),
        "loaded_tensors": len(selected),
        "source_epoch": checkpoint.get("epoch"),
        "quantum_core_initialized_fresh": True,
        "classifier_initialized_fresh": True,
    }


def _backbone_modules(model: D4OrbitClassifier) -> Tuple[torch.nn.Module, ...]:
    """Return every module covered by the classical-backbone checkpoint."""

    modules = [model.physics, model.encoder, model.orbit_projection]
    for optional in (
        model.physics_summary,
        model.physics_summary_norm,
        model.physics_summary_head,
    ):
        if optional is not None:
            modules.append(optional)
    return tuple(modules)


def freeze_backbone(model: D4OrbitClassifier) -> Dict[str, int | bool]:
    """Freeze checkpoint-loaded parameters and stateful module behavior."""

    frozen_parameters = 0
    frozen_tensors = 0
    for name, parameter in model.named_parameters():
        if any(name.startswith(prefix) for prefix in BACKBONE_PREFIXES):
            parameter.requires_grad_(False)
            frozen_parameters += parameter.numel()
            frozen_tensors += 1
    for module in _backbone_modules(model):
        module.eval()
    if frozen_parameters == 0:
        raise RuntimeError("The selected backbone freeze matched no parameters")
    return {
        "backbone_frozen": True,
        "frozen_parameter_tensors": frozen_tensors,
        "frozen_parameters": frozen_parameters,
        "normalization_buffers_frozen": True,
    }


def keep_frozen_backbone_in_eval(model: D4OrbitClassifier) -> None:
    """Undo the recursive mode change from ``model.train()`` for the backbone."""

    for module in _backbone_modules(model):
        module.eval()


def optimizer_parameter_groups(
    model: D4OrbitClassifier,
) -> Tuple[
    List[torch.nn.Parameter],
    List[torch.nn.Parameter],
    List[torch.nn.Parameter],
]:
    encoder_parameters = [
        parameter
        for module in (model.physics, model.encoder)
        for parameter in module.parameters()
        if parameter.requires_grad
    ]
    head_modules = [model.orbit_projection, model.head]
    if model.physics_summary_head is not None:
        head_modules.append(model.physics_summary_head)
    if model.context_projection is not None:
        head_modules.append(model.context_projection)
    head_parameters = [
        parameter
        for module in head_modules
        for parameter in module.parameters()
        if parameter.requires_grad
    ]
    core_parameters = [
        parameter
        for parameter in model.core.parameters()
        if parameter.requires_grad
    ]
    groups = (encoder_parameters, head_parameters, core_parameters)
    grouped_ids = [id(parameter) for group in groups for parameter in group]
    trainable_ids = {
        id(parameter)
        for parameter in model.parameters()
        if parameter.requires_grad
    }
    if len(grouped_ids) != len(set(grouped_ids)):
        raise RuntimeError("A parameter appears in multiple optimizer groups")
    if set(grouped_ids) != trainable_ids:
        raise RuntimeError("Optimizer groups do not cover every trainable parameter")
    return groups


@torch.no_grad()
def evaluate(
    model: D4OrbitClassifier,
    loader,
    device: torch.device,
    class_names: List[str],
) -> Tuple[Dict, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_labels, all_logits, all_indices = [], [], []
    for images, labels, indices in loader:
        images = images.to(device, non_blocking=True).contiguous(
            memory_format=torch.channels_last
        )
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == "cuda",
        ):
            logits = model(images)
        all_labels.append(labels.numpy())
        all_logits.append(logits.float().cpu().numpy())
        all_indices.append(indices.numpy())
    labels = np.concatenate(all_labels)
    logits = np.concatenate(all_logits)
    indices = np.concatenate(all_indices)
    return (
        classification_metrics(labels, logits, class_names),
        labels,
        logits,
        indices,
    )


@torch.no_grad()
def symmetry_audit(
    model: D4OrbitClassifier,
    loader,
    device: torch.device,
    sample_limit: int = 16,
) -> Dict:
    model.eval()
    images = next(iter(loader))[0][:sample_limit]
    images = images.to(device).contiguous(memory_format=torch.channels_last)
    base_logits, base_auxiliary = model(images, return_aux=True)
    audit: Dict[str, Dict] = {}
    all_logit_differences = []
    for element in D4_ELEMENTS:
        logits, auxiliary = model(
            d4_transform(images, *element), return_aux=True
        )
        permutation = right_regular_permutation(element).to(device)
        expected_angles = base_auxiliary["angles"].index_select(
            -1, permutation
        )
        angle_difference = (
            auxiliary["angles"] - expected_angles
        ).abs().float()
        logit_difference = (logits - base_logits).abs().float().reshape(-1)
        all_logit_differences.append(logit_difference)
        record = {
            "angle_regular_max": float(angle_difference.max()),
            "logit_invariant_max": float(logit_difference.max()),
            "logit_invariant_mean": float(logit_difference.mean()),
        }
        if base_auxiliary["equivariant"] is not None:
            for name in ("z", "x"):
                expected = base_auxiliary["equivariant"][name].index_select(
                    -1, permutation
                )
                difference = (
                    auxiliary["equivariant"][name] - expected
                ).abs().float()
                record[f"circuit_{name}_regular_max"] = float(
                    difference.max()
                )
        audit[f"r{element[0]}s{element[1]}"] = record
    combined = torch.cat(all_logit_differences).cpu().numpy()
    audit["summary"] = {
        "max": float(combined.max()),
        "mean": float(combined.mean()),
        "p99": float(np.quantile(combined, 0.99)),
        "samples": int(len(images)),
        "actions": 8,
    }
    return audit


def train(
    config: Config,
    loaders: LoaderBundle,
    stage: StageSpec,
    output_dir: str | Path,
    device: torch.device,
    backbone_checkpoint: str | Path | None = None,
) -> Path:
    """Train one fixed stage and return its selected checkpoint path."""

    output_dir = Path(output_dir)
    if output_dir.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing stage output: {output_dir}"
        )
    output_dir.mkdir(parents=True)
    seed_everything(stage.seed, config.deterministic)
    model = build_model(
        config, core=stage.core, include_context=stage.include_context
    ).to(device=device, memory_format=torch.channels_last)
    expected_parameters = 272_805 if stage.core == "classical" else 245_221
    if config.dataset_id == "model_iv":
        expected_parameters += ModelIVPhysicsSummary.output_dim * 3 + 3
    actual_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    if actual_parameters != expected_parameters:
        raise RuntimeError(
            f"Selected model parameter drift: {actual_parameters} != "
            f"{expected_parameters}"
        )

    initialization = {"mode": "fresh"}
    if backbone_checkpoint:
        if stage.core != "quantum":
            raise ValueError("Backbone initialization is only used by quantum stage")
        initialization = load_backbone_checkpoint(model, backbone_checkpoint)
    if config.freeze_backbone_during_quantum and stage.core == "quantum":
        if not backbone_checkpoint:
            raise ValueError(
                "A checkpoint-initialized quantum stage is required to freeze "
                "the classical backbone"
            )
        initialization.update(freeze_backbone(model))

    class_weights = None

    stage_config = {
        **config.to_dict(),
        "stage_spec": {
            key: getattr(stage, key)
            for key in stage.__dataclass_fields__
        },
        "output_dir": str(output_dir.resolve()),
        "effective_class_weights": (
            class_weights.detach().cpu().tolist()
            if class_weights is not None
            else None
        ),
    }
    _atomic_json(output_dir / "config.json", stage_config)
    _atomic_json(output_dir / "initialization.json", initialization)
    _atomic_json(output_dir / "parameters.json", model.parameter_report())

    encoder_parameters, head_parameters, core_parameters = (
        optimizer_parameter_groups(model)
    )
    optimizer = torch.optim.AdamW(
        (
            {
                "params": encoder_parameters,
                "lr": stage.encoder_learning_rate,
            },
            {"params": head_parameters, "lr": stage.learning_rate},
            {"params": core_parameters, "lr": stage.core_learning_rate},
        ),
        weight_decay=config.weight_decay,
    )
    total_steps = max(1, stage.epochs * len(loaders.train))
    warmup_steps = max(
        1, min(3 * len(loaders.train), int(0.10 * total_steps))
    )

    def learning_rate_factor(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(
            total_steps - warmup_steps, 1
        )
        return 0.01 + 0.99 * 0.5 * (
            1.0 + math.cos(math.pi * progress)
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, learning_rate_factor
    )
    history = []
    best_selection_key = (-math.inf, -math.inf, -math.inf)
    best_epoch = -1
    stale_epochs = 0
    run_start = time.time()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(stage.epochs):
        model.train()
        if config.freeze_backbone_during_quantum and stage.core == "quantum":
            keep_frozen_backbone_in_eval(model)
        epoch_start = time.time()
        loss_sum = 0.0
        correct = 0
        seen = 0
        core_gradient_sum = 0.0
        for images, targets, _ in loaders.train:
            images = images.to(device, non_blocking=True).contiguous(
                memory_format=torch.channels_last
            )
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                logits = model(images)
                loss = F.cross_entropy(
                    logits,
                    targets,
                    weight=class_weights,
                    label_smoothing=config.label_smoothing,
                )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            gradient_squared = 0.0
            for parameter in model.core.parameters():
                if parameter.grad is not None:
                    gradient_squared += float(
                        parameter.grad.detach().float().square().sum()
                    )
            core_gradient_sum += math.sqrt(gradient_squared)
            optimizer.step()
            scheduler.step()

            batch_size = targets.numel()
            seen += batch_size
            loss_sum += float(loss.detach()) * batch_size
            correct += int((logits.argmax(dim=1) == targets).sum())

        metrics, labels, logits, indices = evaluate(
            model,
            loaders.validation,
            device,
            loaders.class_names,
        )
        selection_key = (
            metrics["balanced_accuracy"],
            metrics["macro_auc_ovr"],
            -metrics["nll"],
        )
        record = {
            "epoch": epoch + 1,
            "train_loss": loss_sum / seen,
            "train_accuracy": correct / seen,
            "validation": metrics,
            "encoder_learning_rate": optimizer.param_groups[0]["lr"],
            "learning_rate": optimizer.param_groups[1]["lr"],
            "core_learning_rate": optimizer.param_groups[2]["lr"],
            "mean_core_gradient_norm": core_gradient_sum
            / max(len(loaders.train), 1),
            "epoch_seconds": time.time() - epoch_start,
            "gpu_peak_memory_bytes": (
                int(torch.cuda.max_memory_allocated(device))
                if device.type == "cuda"
                else 0
            ),
            "selection_key": list(selection_key),
        }
        history.append(record)
        _atomic_json(output_dir / "history.json", history)
        _atomic_checkpoint(
            output_dir / "last.pt",
            {"model": model.state_dict(), "epoch": epoch + 1, "record": record},
        )
        print(f"EPOCH {json.dumps(record, sort_keys=True)}", flush=True)

        if selection_key > best_selection_key:
            best_selection_key = selection_key
            best_epoch = epoch + 1
            stale_epochs = 0
            _atomic_checkpoint(
                output_dir / "best.pt",
                {
                    "model": model.state_dict(),
                    "epoch": best_epoch,
                    "record": record,
                },
            )
            np.savez_compressed(
                output_dir / "best_validation_predictions.npz",
                indices=indices,
                labels=labels,
                logits=logits,
            )
        else:
            stale_epochs += 1
            if stale_epochs >= stage.patience:
                print(
                    f"EARLY_STOP epoch={epoch + 1} best_epoch={best_epoch}",
                    flush=True,
                )
                break

    best_checkpoint = output_dir / "best.pt"
    checkpoint = torch.load(
        best_checkpoint, map_location=device, weights_only=False
    )
    model.load_state_dict(checkpoint["model"], strict=True)
    final_metrics, final_labels, final_logits, _ = evaluate(
        model, loaders.validation, device, loaders.class_names
    )
    symmetry = symmetry_audit(model, loaders.validation, device)
    _atomic_json(output_dir / "symmetry_audit.json", symmetry)
    summary = {
        "stage": stage.name,
        "best_epoch": best_epoch,
        "validation": final_metrics,
        "parameters": model.parameter_report(),
        "symmetry": symmetry["summary"],
        "initialization": initialization,
        "wall_seconds": time.time() - run_start,
        "official_test_evaluated": False,
    }
    _atomic_json(output_dir / "summary.json", summary)
    _write_validation_artifacts(
        output_dir,
        stage,
        best_epoch,
        final_metrics,
        final_labels,
        final_logits,
        loaders.class_names,
    )
    print(f"SUMMARY {json.dumps(summary, sort_keys=True)}", flush=True)
    return best_checkpoint


## 9. Architecture, gradient, and D4 preflight

This small data-free check validates parameter counts, tensor shapes,
TorchQuantum forward/backward behavior, input gradients, circuit-parameter
gradients, and final-logit invariance under all eight D4 actions.


In [11]:
# This cell is intentionally data-free: it catches architecture drift and backend
# autograd failures before any long training run.
verification_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
require_torchquantum()
backend_report = smoke_test_torchquantum(verification_device)

classical_model = build_model(config, core="classical", include_context=True)
quantum_model = build_model(config, core="quantum", include_context=False).to(verification_device)
expected_classical = 276_993 if config.dataset_id == "model_iv" else 272_805
expected_quantum = 249_409 if config.dataset_id == "model_iv" else 245_221
actual_classical = sum(p.numel() for p in classical_model.parameters() if p.requires_grad)
actual_quantum = sum(p.numel() for p in quantum_model.parameters() if p.requires_grad)
assert actual_classical == expected_classical, (actual_classical, expected_classical)
assert actual_quantum == expected_quantum, (actual_quantum, expected_quantum)
assert quantum_model.core.params.numel() == 88
assert quantum_model.core.output_dim == 48
assert len(D4_ELEMENTS) == 8
assert len(R_EDGES) == 8 and len(R2_EDGES) == 4 and len(S_EDGES) == 4

quantum_model.eval()
probe = torch.linspace(
    0.0, 1.0, config.image_size * config.image_size,
    device=verification_device,
).reshape(1, 1, config.image_size, config.image_size)
probe.requires_grad_(True)
logits, auxiliary = quantum_model(probe, return_aux=True)
assert logits.shape == (1, 3)
assert auxiliary["encoded"].shape == (1, 8, 128)
assert auxiliary["angles"].shape == (1, 4, 2, 8)
assert auxiliary["invariants"].shape == (1, 48)
logits.square().mean().backward()
assert probe.grad is not None and torch.isfinite(probe.grad).all()
assert quantum_model.core.params.grad is not None
assert torch.isfinite(quantum_model.core.params.grad).all()

with torch.no_grad():
    reference = quantum_model(probe.detach())
    d4_errors = {}
    for element in D4_ELEMENTS:
        transformed = quantum_model(d4_transform(probe.detach(), *element))
        d4_errors[f"r{element[0]}s{element[1]}"] = float(
            (transformed - reference).abs().max()
        )
assert max(d4_errors.values()) < 2e-4, d4_errors

print({
    "backend": backend_report,
    "classical_parameters": actual_classical,
    "quantum_parameters": actual_quantum,
    "circuit_parameters": quantum_model.core.params.numel(),
    "invariant_features": quantum_model.core.output_dim,
    "max_d4_logit_error": max(d4_errors.values()),
})
del classical_model, quantum_model, probe, logits, auxiliary
if torch.cuda.is_available():
    torch.cuda.empty_cache()


{'backend': {'backend': 'torchquantum', 'device': 'cuda', 'quantum_parameters': 88, 'invariant_features': 48}, 'classical_parameters': 276993, 'quantum_parameters': 249409, 'circuit_parameters': 88, 'invariant_features': 48, 'max_d4_logit_error': 0.00017020106315612793}


## 10. Two-stage training

Training requires CUDA. Stage 1 performs 18-epoch classical-context
pretraining. Stage 2 rebuilds loaders with its own deterministic shuffle,
creates the selected 50-epoch quantum model, and passes the newly selected
Stage-1 checkpoint into the shared backbone. The held-out test plan is only
printed; it is never evaluated here. `FINAL_TEST_ONLY` mode skips this cell's
training body so a completed run can be evaluated after a kernel restart.


In [12]:
if FINAL_TEST_ONLY:
    print("Two-stage training skipped: completed run opened for final test only.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("The full D4-ORQB training workflow requires a CUDA-capable GPU")
    device = torch.device("cuda")
    print("QUANTUM_BACKEND", smoke_test_torchquantum(device))
    torch.cuda.empty_cache()

    # Stage 1: parameter-matched classical-context pretraining.
    pretraining = pretrain_spec(config)
    pretrain_loaders, pretrain_test_plan = build_notebook_training_loaders(
        config,
        pretraining.seed,
        device,
        TEST_ROOT,
        TEST_FRACTION,
    )
    _atomic_json(
        config.output_path / "data_partitions.json",
        pretrain_loaders.metadata,
    )
    print("Fixed partition provenance:")
    print("Validation policy:", pretrain_loaders.metadata["validation_mode"])
    print("Test policy:      ", pretrain_loaders.metadata["test_mode"])
    print(json.dumps(pretrain_loaders.metadata["partition_counts"], indent=2, sort_keys=True))
    print("Held-out test policy:", pretrain_test_plan.description)
    backbone_checkpoint = train(
        config,
        pretrain_loaders,
        pretraining,
        config.output_path / pretraining.name,
        device,
    )

    # Stage 2: rebuild deterministic loaders with the quantum seed, instantiate a fresh
    # quantum core/main head, and load the source-defined checkpoint prefixes. For
    # Model IV those prefixes intentionally include its auxiliary physics-summary head.
    quantum = quantum_spec(config)
    quantum_loaders, final_test_plan = build_notebook_training_loaders(
        config,
        quantum.seed,
        device,
        TEST_ROOT,
        TEST_FRACTION,
    )
    quantum_checkpoint = train(
        config,
        quantum_loaders,
        quantum,
        config.output_path / quantum.name,
        device,
        backbone_checkpoint=backbone_checkpoint,
    )

    print("Selected pretraining checkpoint:", backbone_checkpoint)
    print("Selected quantum checkpoint:    ", quantum_checkpoint)
    print("Held-out test policy:           ", final_test_plan.description)
    print("The test set has not been evaluated by this training cell.")


QUANTUM_BACKEND {'backend': 'torchquantum', 'device': 'cuda', 'quantum_parameters': 88, 'invariant_features': 48}


CACHE_PROGRESS 384/54702


CACHE_PROGRESS 768/54702


CACHE_PROGRESS 1152/54702


CACHE_PROGRESS 1536/54702


...

EPOCH {"core_learning_rate": 7.208321681691943e-05, "encoder_learning_rate": 7.208321681691943e-06, "epoch": 48, "epoch_seconds": 46.025819063186646, "gpu_peak_memory_bytes": 9225915392, "learning_rate": 4.324993009015166e-05, "mean_core_gradient_norm": 0.000389824716746876, "selection_key": [0.3465200255305569, 0.5148818024413555, -1.10080087184906], "train_accuracy": 0.3812934167795772, "train_loss": 1.0900599239108244, "validation": {"accuracy": 0.34833305961570044, "balanced_accuracy": 0.3465200255305569, "brier": 0.6680322323131174, "confusion_matrix": [[532, 884, 584], [470, 983, 636], [507, 887, 606]], "ece_15": 0.016182846962514805, "macro_auc_ovr": 0.5148818024413555, "macro_f1": 0.3419823149109364, "nll": 1.10080087184906, "per_class": {"axion": {"auc_ovr": 0.5230734287111763, "f1": 0.30322029068110573, "precision": 0.35255135851557323, "recall": 0.266, "support": 2000}, "cdm": {"auc_ovr": 0.5243868477740545, "f1": 0.4059467272351848, "precision": 0.35693536673928833, "recall

EPOCH {"core_learning_rate": 5.552697540769592e-05, "encoder_learning_rate": 5.552697540769592e-06, "epoch": 49, "epoch_seconds": 45.725900173187256, "gpu_peak_memory_bytes": 9225915392, "learning_rate": 3.3316185244617555e-05, "mean_core_gradient_norm": 0.0003794153728396997, "selection_key": [0.3468054092867401, 0.5143555409550147, -1.1007550954818726], "train_accuracy": 0.37909972686409876, "train_loss": 1.0901221148128917, "validation": {"accuracy": 0.35013959599277383, "balanced_accuracy": 0.3468054092867401, "brier": 0.6680076229558017, "confusion_matrix": [[579, 1053, 368], [506, 1201, 382], [556, 1092, 352]], "ece_15": 0.015998618099407173, "macro_auc_ovr": 0.5143555409550147, "macro_f1": 0.3289817232893763, "nll": 1.1007550954818726, "per_class": {"axion": {"auc_ovr": 0.5224762778185376, "f1": 0.31804449327107936, "precision": 0.35283363802559414, "recall": 0.2895, "support": 2000}, "cdm": {"auc_ovr": 0.5243435854475825, "f1": 0.44195032198712053, "precision": 0.35893604303646

EPOCH {"core_learning_rate": 5e-05, "encoder_learning_rate": 5e-06, "epoch": 50, "epoch_seconds": 45.446184158325195, "gpu_peak_memory_bytes": 9225915392, "learning_rate": 3e-05, "mean_core_gradient_norm": 0.00037786634182521844, "selection_key": [0.3485852880165949, 0.5143749227437638, -1.1009464263916016], "train_accuracy": 0.3813579370712089, "train_loss": 1.0899850472836408, "validation": {"accuracy": 0.3521103629495812, "balanced_accuracy": 0.3485852880165949, "brier": 0.6681254982869691, "confusion_matrix": [[682, 1073, 245], [603, 1232, 254], [649, 1121, 230]], "ece_15": 0.01543578267156209, "macro_auc_ovr": 0.5143749227437638, "macro_f1": 0.3206874372685661, "nll": 1.1009464263916016, "per_class": {"axion": {"auc_ovr": 0.5225383345561262, "f1": 0.3467208947635994, "precision": 0.35263702171664946, "recall": 0.341, "support": 2000}, "cdm": {"auc_ovr": 0.524683401148875, "f1": 0.44678150498640073, "precision": 0.35960303561004087, "recall": 0.5897558640497846, "support": 2089}, "

SUMMARY {"best_epoch": 27, "initialization": {"checkpoint": "<runtime-root>/outputs/model_iv/pretrain_context/best.pt", "classifier_initialized_fresh": true, "loaded_prefixes": ["physics.", "physics_summary.", "physics_summary_norm.", "physics_summary_head.", "encoder.", "orbit_projection."], "loaded_tensors": 114, "quantum_core_initialized_fresh": true, "source_epoch": 6}, "official_test_evaluated": false, "parameters": {"core": 88, "core_architecture": "quantum", "encoder": 242338, "encoder_output_dim": 128, "encoder_variant": "tiny", "execution_backend": "torchquantum", "head_and_context": 1763, "input_channels": 8, "morphology_channels": 0, "morphology_variant": "model_iv_sis_closure", "observable_readout": "pair", "orbit_projection": 1032, "physics_summary_dim": 1395, "physics_summary_head": 4188, "quantum_encoding": "angle", "total": 249409}, "stage": "quantum_seed2_50ep", "symmetry": {"actions": 8, "max": 0.002153158187866211, "mean": 0.0003162166103720665, "p99": 0.002065286738

Selected pretraining checkpoint: <runtime-root>/outputs/model_iv/pretrain_context/best.pt
Selected quantum checkpoint:     <runtime-root>/outputs/model_iv/quantum_seed2_50ep/best.pt
Held-out test policy:            15% class-stratified holdout carved from Model-IV train/
The test set has not been evaluated by this training cell.


## 11. Final held-out reporting utilities

These helpers compute the same complete metric family as validation and
save ROC/confusion plots. They reload `best.pt` and refuse to overwrite an
existing `final_test/` result, making accidental repeated evaluation visible.


In [13]:
def _save_final_test_plots(
    output_dir: Path,
    labels: np.ndarray,
    logits: np.ndarray,
    class_names: List[str],
    metrics: Dict,
) -> None:
    import matplotlib.pyplot as plt

    shifted = logits - logits.max(axis=1, keepdims=True)
    exponent = np.exp(shifted)
    probabilities = exponent / exponent.sum(axis=1, keepdims=True)

    figure, axis = plt.subplots(figsize=(6.4, 5.2))
    for class_index, class_name in enumerate(class_names):
        false_positive_rate, true_positive_rate = _binary_roc_curve(
            labels == class_index, probabilities[:, class_index]
        )
        if false_positive_rate.size:
            auc = metrics["per_class"][class_name]["auc_ovr"]
            axis.plot(false_positive_rate, true_positive_rate, lw=2, label=f"{class_name} (AUC={auc:.4f})")
    axis.plot((0, 1), (0, 1), "k--", lw=1, label="Chance")
    axis.set(xlim=(0, 1), ylim=(0, 1), xlabel="False positive rate", ylabel="True positive rate", title="Final held-out one-vs-rest ROC")
    axis.grid(alpha=0.25)
    axis.legend(loc="lower right")
    figure.tight_layout()
    figure.savefig(output_dir / "roc_curve.png", dpi=160)
    plt.close(figure)

    matrix = np.asarray(metrics["confusion_matrix"])
    figure, axis = plt.subplots(figsize=(5.6, 5.0))
    image = axis.imshow(matrix, cmap="Blues")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center")
    axis.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted class",
        ylabel="True class",
        title="Final held-out confusion matrix",
    )
    figure.colorbar(image, ax=axis)
    figure.tight_layout()
    figure.savefig(output_dir / "confusion_matrix.png", dpi=160)
    plt.close(figure)


def evaluate_final_test_once(
    config: Config,
    plan: HeldoutTestPlan,
    checkpoint_path: str | Path,
    device: torch.device,
) -> Dict:
    """Reload the validation-selected checkpoint and write final test artifacts once."""

    output_dir = config.output_path / "final_test"
    if output_dir.exists():
        raise FileExistsError(f"Final test output already exists: {output_dir}")

    # Validate/materialize the loader and checkpoint before reserving the final
    # artifact path. A blank or mistyped official TEST_ROOT must not poison retries.
    loader = build_final_test_loader(config, plan, device)
    model = build_model(config, core="quantum", include_context=False).to(
        device=device, memory_format=torch.channels_last
    )
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"], strict=True)
    metrics, labels, logits, indices = evaluate(model, loader, device, plan.class_names)
    result = {
        "dataset_id": config.dataset_id,
        "checkpoint": str(Path(checkpoint_path).resolve()),
        "checkpoint_epoch": checkpoint.get("epoch"),
        "test_kind": plan.kind,
        "test_description": plan.description,
        "used_for_model_selection": False,
        "metrics": metrics,
    }

    temporary_dir = output_dir.with_name(
        f".final_test-building-{os.getpid()}-{time.time_ns()}"
    )
    temporary_dir.mkdir(parents=False)
    _atomic_json(temporary_dir / "metrics.json", result)
    np.savez_compressed(
        temporary_dir / "predictions.npz",
        indices=indices,
        labels=labels,
        logits=logits,
    )
    _save_final_test_plots(temporary_dir, labels, logits, plan.class_names, metrics)
    lines = [
        "# Final held-out evaluation",
        "",
        f"- Dataset: `{config.dataset_id}`",
        f"- Test kind: `{plan.kind}`",
        f"- Provenance: {plan.description}",
        f"- Samples: {metrics['samples']}",
        f"- Accuracy: {metrics['accuracy']:.6f}",
        f"- Balanced accuracy: {metrics['balanced_accuracy']:.6f}",
        f"- Macro F1: {metrics['macro_f1']:.6f}",
        f"- Macro one-vs-rest AUC: {metrics['macro_auc_ovr']:.6f}",
        "- Used for checkpoint selection: **No**",
        "",
    ]
    _atomic_text(temporary_dir / "README.md", "\n".join(lines))
    os.replace(temporary_dir, output_dir)
    return result


## 12. Explicit one-time final test evaluation

Leave this skipped during development. After all choices are frozen and
validation evidence is accepted, set `CONFIRM_FINAL_TEST_EVALUATION = True`
and execute this cell once. Models I-III report an official test; Models
IV-V report a carved development holdout and never call it official.


In [14]:
# This explicit confirmation protects Models I-III's official test set and prevents
# accidental run-all evaluation. Models IV/V are labeled carved holdouts, never official.
if not CONFIRM_FINAL_TEST_EVALUATION:
    print(
        "FINAL TEST SKIPPED. Review validation results, then set "
        "CONFIRM_FINAL_TEST_EVALUATION = True and rerun this cell once."
    )
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    quantum_stage = quantum_spec(config)
    selected_checkpoint = config.output_path / quantum_stage.name / "best.pt"
    if not selected_checkpoint.is_file():
        raise FileNotFoundError(
            f"Validation-selected quantum checkpoint is missing: {selected_checkpoint}"
        )
    # Reconstruct the same fixed split/test plan without running either training stage.
    _, final_test_plan = build_notebook_training_loaders(
        config,
        quantum_stage.seed,
        device,
        TEST_ROOT,
        TEST_FRACTION,
    )
    final_test_result = evaluate_final_test_once(
        config,
        final_test_plan,
        selected_checkpoint,
        device,
    )
    print(json.dumps(final_test_result, indent=2, sort_keys=True))


CACHE_READY <runtime-root>/cache/model_iv_96 samples=54702


CACHE_READY <runtime-root>/cache/model_iv_96_validation samples=6089


{
  "checkpoint": "<runtime-root>/outputs/model_iv/quantum_seed2_50ep/best.pt",
  "checkpoint_epoch": 27,
  "dataset_id": "model_iv",
  "metrics": {
    "accuracy": 0.3595368677635588,
    "balanced_accuracy": 0.35543539974912525,
    "brier": 0.6670120113989512,
    "confusion_matrix": [
      [
        932,
        1638,
        130
      ],
      [
        812,
        1896,
        97
      ],
      [
        911,
        1667,
        122
      ]
    ],
    "ece_15": 0.01481037968763996,
    "macro_auc_ovr": 0.5274838375623402,
    "macro_f1": 0.30058563518770487,
    "nll": 1.0992649793624878,
    "per_class": {
      "axion": {
        "auc_ovr": 0.5274881757324991,
        "f1": 0.3480859010270775,
        "precision": 0.35103578154425613,
        "recall": 0.3451851851851852,
        "support": 2700
      },
      "cdm": {
        "auc_ovr": 0.5486840958605664,
        "f1": 0.47364476642518116,
        "precision": 0.3645452797538935,
        "recall": 0.6759358288770053,
   

## 13. Run review and checkpoint candidate

The notebook leaves all generated state in ignored runtime output. Review
provenance, split counts, validation metrics, the eight-action symmetry
audit, and final held-out results before deliberately documenting and
promoting any checkpoint.


In [15]:
quantum_stage = quantum_spec(config)
stage_dir = config.output_path / quantum_stage.name
expected = {
    "best_checkpoint": stage_dir / "best.pt",
    "validation_metrics": stage_dir / "validation_metrics.md",
    "validation_roc": stage_dir / "validation_roc_curve.png",
    "symmetry_audit": stage_dir / "symmetry_audit.json",
    "final_test_metrics": config.output_path / "final_test" / "metrics.json",
}
for name, path in expected.items():
    print(f"{name:24s} {'READY' if path.exists() else 'not generated'}  {path}")

print(
    "\nThe checkpoint remains in the ignored run directory. Promote nothing to "
    "weights/ until validation, symmetry, provenance, and (for the final selected "
    "run) held-out evidence have been reviewed and documented."
)


best_checkpoint          READY  <runtime-root>/outputs/model_iv/quantum_seed2_50ep/best.pt
validation_metrics       READY  <runtime-root>/outputs/model_iv/quantum_seed2_50ep/validation_metrics.md
validation_roc           READY  <runtime-root>/outputs/model_iv/quantum_seed2_50ep/validation_roc_curve.png
symmetry_audit           READY  <runtime-root>/outputs/model_iv/quantum_seed2_50ep/symmetry_audit.json
final_test_metrics       READY  <runtime-root>/outputs/model_iv/final_test/metrics.json

The checkpoint remains in the ignored run directory. Promote nothing to weights/ until validation, symmetry, provenance, and (for the final selected run) held-out evidence have been reviewed and documented.
